1. IMPORT KONFIGURASI

In [27]:
# ==========================================
# IMPORT & KONFIGURASI DASAR
# ==========================================

import os
import time
from pathlib import Path
from datetime import datetime, timedelta, timezone

import pandas as pd
from apify_client import ApifyClient

print("Library berhasil di-import.")

Library berhasil di-import.


2. PATH DATA

In [28]:
# ==========================================
# PATH DATA PENELITIAN
# ==========================================

from pathlib import Path

# Gunakan folder BARU agar checkpoint Palu lama
# tidak tercampur dengan pengambilan data penelitian baru.
BASE_DIR = Path(
    r"C:\Project Mariana\data_bencana_nasional"
)

RAW_DIR = BASE_DIR / "raw"
CLEAN_DIR = BASE_DIR / "clean"

CHECKPOINT_FILE = BASE_DIR / "checkpoint.csv"
FAILED_FILE = BASE_DIR / "failed_windows.csv"

# Buat folder jika belum ada
RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLEAN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("BASE_DIR :", BASE_DIR)
print("RAW_DIR  :", RAW_DIR)
print("CLEAN_DIR:", CLEAN_DIR)
print("CHECKPOINT:", CHECKPOINT_FILE)
print("FAILED    :", FAILED_FILE)

BASE_DIR : C:\Project Mariana\data_bencana_nasional
RAW_DIR  : C:\Project Mariana\data_bencana_nasional\raw
CLEAN_DIR: C:\Project Mariana\data_bencana_nasional\clean
CHECKPOINT: C:\Project Mariana\data_bencana_nasional\checkpoint.csv
FAILED    : C:\Project Mariana\data_bencana_nasional\failed_windows.csv


3. KEYWORD

In [29]:
# ============================================================
# KONFIGURASI KASUS PENELITIAN
# ============================================================

DISASTER_CASES = {

    # ========================================================
    # 1. LOMBOK 2018
    # ========================================================
    "lombok_2018": {

        "label": "Gempa Lombok 2018",

        "start_date": "2018-07-29",
        "end_date": "2018-08-31",

        "event_terms": [
            "Lombok",
            "gempa Lombok",
            "gempa bumi Lombok",
            "gempa NTB",
            "gempa bumi NTB",
            "NTB",
            "Nusa Tenggara Barat",
            "Mataram",
            "Lombok Utara",
            "Lombok Barat",
            "Lombok Timur",
            "Lombok Tengah"
        ],

        "policy_terms": [
            "bencana nasional",
            "status bencana nasional",
            "status nasional",
            "penetapan bencana nasional",
            "penetapan status bencana",
            "penetapan status",
            "status bencana",
            "status darurat",
            "bencana daerah",
            "status bencana daerah",
            "ditetapkan sebagai bencana nasional",
            "ditetapkan bencana nasional",
            "tetapkan bencana nasional",
            "menetapkan bencana nasional",
            "jadi bencana nasional",
            "jadiin bencana nasional"
        ]
    },


    # ========================================================
    # 2. PALU 2018
    # ========================================================
    "palu_2018": {

        "label": "Gempa & Tsunami Palu 2018",

        "start_date": "2018-09-28",
        "end_date": "2018-10-31",

        "event_terms": [
            "Palu",
            "gempa Palu",
            "gempa bumi Palu",
            "tsunami Palu",
            "gempa tsunami Palu",
            "Donggala",
            "gempa Donggala",
            "tsunami Donggala",
            "Sulawesi Tengah",
            "gempa Sulawesi Tengah",
            "tsunami Sulawesi Tengah",
            "bencana Palu",
            "bencana Sulawesi Tengah"
        ],

        "policy_terms": [
            "bencana nasional",
            "status bencana nasional",
            "status nasional",
            "penetapan bencana nasional",
            "penetapan status bencana",
            "penetapan status",
            "status bencana",
            "status darurat",
            "bencana daerah",
            "status bencana daerah",
            "ditetapkan sebagai bencana nasional",
            "ditetapkan bencana nasional",
            "tetapkan bencana nasional",
            "menetapkan bencana nasional",
            "jadi bencana nasional",
            "bantuan internasional"
        ]
    },


    # ========================================================
    # 3. SUMATRA 2025
    # ========================================================
    "sumatra_2025": {

        "label": "Banjir & Longsor Sumatra 2025",

        "start_date": "2025-11-26",
        "end_date": "2025-12-31",

        "event_terms": [
            "Aceh",
            "banjir Aceh",
            "longsor Aceh",
            "bencana Aceh",

            "Sumatra Utara",
            "Sumatera Utara",
            "banjir Sumatra Utara",
            "banjir Sumatera Utara",
            "longsor Sumatra Utara",
            "longsor Sumatera Utara",
            "bencana Sumatra Utara",
            "bencana Sumatera Utara",

            "Sumatra Barat",
            "Sumatera Barat",
            "banjir Sumatra Barat",
            "banjir Sumatera Barat",
            "longsor Sumatra Barat",
            "longsor Sumatera Barat",
            "bencana Sumatra Barat",
            "bencana Sumatera Barat",

            "banjir Sumatra",
            "banjir Sumatera",
            "longsor Sumatra",
            "longsor Sumatera",
            "bencana Sumatra",
            "bencana Sumatera"
        ],

        "policy_terms": [
            "bencana nasional",
            "status bencana nasional",
            "status nasional",
            "penetapan bencana nasional",
            "penetapan status bencana",
            "penetapan status",
            "status bencana",
            "status darurat",
            "bencana daerah",
            "status bencana daerah",
            "ditetapkan sebagai bencana nasional",
            "ditetapkan bencana nasional",
            "tetapkan bencana nasional",
            "menetapkan bencana nasional",
            "jadi bencana nasional",

            "prioritas nasional",
            "penanganan nasional",
            "penanganan berskala nasional",
            "berskala nasional",
            "ditangani pemerintah pusat",
            "penanganan pemerintah pusat",
            "pemerintah pusat",
            "status daerah",
            "status provinsi"
        ]
    }
}


# ============================================================
# QUERY BUILDER
# ============================================================

def build_or_group(terms):
    """
    Membuat:
        ("term 1" OR "term 2" OR ...)
    """

    return "(" + " OR ".join(
        f'"{term}"'
        for term in terms
    ) + ")"


def build_research_query(
    event_terms,
    policy_terms
):
    """
    Query utama penelitian:

        EVENT
        AND
        POLICY
    """

    event_group = build_or_group(
        event_terms
    )

    policy_group = build_or_group(
        policy_terms
    )

    return (
        f"{event_group} AND {policy_group}"
    )


# ============================================================
# BANGUN QUERY UNTUK SETIAP KASUS
# ============================================================

for case_id, config in DISASTER_CASES.items():

    config["query"] = build_research_query(
        config["event_terms"],
        config["policy_terms"]
    )


# ============================================================
# TAMPILKAN KONFIGURASI
# ============================================================

print("=" * 80)
print("KONFIGURASI KASUS PENELITIAN")
print("=" * 80)

for case_id, config in DISASTER_CASES.items():

    print()
    print(f"CASE   : {case_id}")
    print(f"LABEL  : {config['label']}")
    print(
        f"PERIOD : "
        f"{config['start_date']} → "
        f"{config['end_date']}"
    )

    print(
        "EVENT TERMS :",
        len(config["event_terms"])
    )

    print(
        "POLICY TERMS:",
        len(config["policy_terms"])
    )

    print("QUERY:")
    print(config["query"])

KONFIGURASI KASUS PENELITIAN

CASE   : lombok_2018
LABEL  : Gempa Lombok 2018
PERIOD : 2018-07-29 → 2018-08-31
EVENT TERMS : 12
POLICY TERMS: 16
QUERY:
("Lombok" OR "gempa Lombok" OR "gempa bumi Lombok" OR "gempa NTB" OR "gempa bumi NTB" OR "NTB" OR "Nusa Tenggara Barat" OR "Mataram" OR "Lombok Utara" OR "Lombok Barat" OR "Lombok Timur" OR "Lombok Tengah") AND ("bencana nasional" OR "status bencana nasional" OR "status nasional" OR "penetapan bencana nasional" OR "penetapan status bencana" OR "penetapan status" OR "status bencana" OR "status darurat" OR "bencana daerah" OR "status bencana daerah" OR "ditetapkan sebagai bencana nasional" OR "ditetapkan bencana nasional" OR "tetapkan bencana nasional" OR "menetapkan bencana nasional" OR "jadi bencana nasional" OR "jadiin bencana nasional")

CASE   : palu_2018
LABEL  : Gempa & Tsunami Palu 2018
PERIOD : 2018-09-28 → 2018-10-31
EVENT TERMS : 13
POLICY TERMS: 16
QUERY:
("Palu" OR "gempa Palu" OR "gempa bumi Palu" OR "tsunami Palu" OR "gempa

In [30]:
# ============================================================
# QUERY BATCHING
# ============================================================

def chunk_list(items, chunk_size):
    """
    Membagi list menjadi beberapa batch.
    """

    return [
        items[i:i + chunk_size]
        for i in range(
            0,
            len(items),
            chunk_size
        )
    ]


def build_query_batches(
    event_terms,
    policy_terms,
    event_chunk_size=5,
    policy_chunk_size=6
):
    """
    Membuat beberapa query agar tidak terlalu panjang.

    Setiap query tetap memiliki:

        EVENT TERMS
        AND
        POLICY TERMS
    """

    event_batches = chunk_list(
        event_terms,
        event_chunk_size
    )

    policy_batches = chunk_list(
        policy_terms,
        policy_chunk_size
    )

    queries = []

    for event_batch in event_batches:

        for policy_batch in policy_batches:

            query = build_research_query(
                event_batch,
                policy_batch
            )

            queries.append(query)

    # Hapus duplicate query
    queries = list(
        dict.fromkeys(queries)
    )

    return queries


# ============================================================
# BANGUN QUERY BATCH UNTUK SEMUA KASUS
# ============================================================

for case_id, config in DISASTER_CASES.items():

    config["queries"] = build_query_batches(
        config["event_terms"],
        config["policy_terms"]
    )


# ============================================================
# LAPORAN QUERY
# ============================================================

print()
print("=" * 80)
print("QUERY BATCH PENELITIAN")
print("=" * 80)

for case_id, config in DISASTER_CASES.items():

    print()
    print(
        f"{case_id}: "
        f"{len(config['queries'])} query"
    )

    for i, query in enumerate(
        config["queries"],
        start=1
    ):

        print()
        print(f"QUERY {i}:")
        print(query)


QUERY BATCH PENELITIAN

lombok_2018: 9 query

QUERY 1:
("Lombok" OR "gempa Lombok" OR "gempa bumi Lombok" OR "gempa NTB" OR "gempa bumi NTB") AND ("bencana nasional" OR "status bencana nasional" OR "status nasional" OR "penetapan bencana nasional" OR "penetapan status bencana" OR "penetapan status")

QUERY 2:
("Lombok" OR "gempa Lombok" OR "gempa bumi Lombok" OR "gempa NTB" OR "gempa bumi NTB") AND ("status bencana" OR "status darurat" OR "bencana daerah" OR "status bencana daerah" OR "ditetapkan sebagai bencana nasional" OR "ditetapkan bencana nasional")

QUERY 3:
("Lombok" OR "gempa Lombok" OR "gempa bumi Lombok" OR "gempa NTB" OR "gempa bumi NTB") AND ("tetapkan bencana nasional" OR "menetapkan bencana nasional" OR "jadi bencana nasional" OR "jadiin bencana nasional")

QUERY 4:
("NTB" OR "Nusa Tenggara Barat" OR "Mataram" OR "Lombok Utara" OR "Lombok Barat") AND ("bencana nasional" OR "status bencana nasional" OR "status nasional" OR "penetapan bencana nasional" OR "penetapan statu

4. APIFY CLIENT

In [ ]:
import os

os.environ["APIFY_API_TOKEN"] = "Masukkan Token API Anda di sini"

print("APIFY_API_TOKEN berhasil diset.")

APIFY_API_TOKEN berhasil diset.


In [32]:
# ==========================================
# APIFY CONFIGURATION
# ==========================================

import os

TOKEN = os.getenv(
    "APIFY_API_TOKEN"
)

if not TOKEN:

    raise RuntimeError(
        "APIFY_API_TOKEN belum tersedia.\n"
        "Pastikan token Apify sudah diset di environment."
    )


# ============================================================
# APIFY CLIENT
# ============================================================

client = ApifyClient(
    TOKEN
)


# ============================================================
# ACTOR
# ============================================================

ACTOR_ID = "CJdippxWmn9uRfooo"


print(
    "APIFY API TOKEN : tersedia"
)

print(
    "Token prefix    :",
    TOKEN[:8] + "..."
)

print(
    "Actor ID        :",
    ACTOR_ID
)

APIFY API TOKEN : tersedia
Token prefix    : apify_ap...
Actor ID        : CJdippxWmn9uRfooo


In [33]:
# ==========================================
# CEK ACTOR APIFY
# ==========================================

print("=" * 80)
print("CEK ACTOR APIFY")
print("=" * 80)

print("ACTOR_ID :", ACTOR_ID)

try:
    actor = client.actor(ACTOR_ID)

    print("Actor object :", actor)
    print("Actor ID     :", ACTOR_ID)

    # Tes apakah Actor benar-benar bisa dipanggil
    print()
    print("Menguji akses Actor...")

    test_run = actor.call(
        run_input={
            "searchTerms": ["Palu"],
            "maxItems": 1
        }
    )

    print()
    print("✅ ACTOR BERHASIL DIPANGGIL")
    print("Dataset ID :", test_run.default_dataset_id)

except Exception as e:
    print()
    print("❌ ACTOR GAGAL DIPANGGIL")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

CEK ACTOR APIFY
ACTOR_ID : CJdippxWmn9uRfooo
Actor object : <apify_client._resource_clients.actor.ActorClient object at 0x000001BB85A81550>
Actor ID     : CJdippxWmn9uRfooo

Menguji akses Actor...


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:ssPPUcQdXSrXxVXaz] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:ssPPUcQdXSrXxVXaz] -> 2026-10-10T07:59:47.655Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:ssPPUcQdXSrXxVXaz] -> 2026-10-10T07:59:47.657Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:ssPPUcQdXSrXxVXaz] -> 2026-10-10T07:59:47.728Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:ssPPUcQdXSrXxVXaz] -> 2026-10-10T07:59:47.730Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:ssPPUcQdXSrXxVXaz] -> 2026-10-10T07:59:54.119Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru


✅ ACTOR BERHASIL DIPANGGIL
Dataset ID : LUAYpIfq1mQQxTDqE


5. WIB -> UNIX

In [34]:
# ==========================================
# WIB → UNIX
# ==========================================

WIB = timezone(timedelta(hours=7))


def wib_to_unix(timestamp_string):
    """
    Mengubah waktu WIB:
    YYYY-MM-DD HH:MM:SS

    menjadi Unix timestamp UTC.
    """

    dt_wib = datetime.strptime(
        timestamp_string,
        "%Y-%m-%d %H:%M:%S"
    ).replace(tzinfo=WIB)

    dt_utc = dt_wib.astimezone(timezone.utc)

    return int(dt_utc.timestamp())


# TEST
test_timestamp = "2018-09-28 00:00:00"

print(
    test_timestamp,
    "WIB =",
    wib_to_unix(test_timestamp)
)

2018-09-28 00:00:00 WIB = 1538067600


6. FUNGSI ADAPTIVE SCRAPER

In [35]:
# ==========================================
# ADAPTIVE SCRAPER CONFIG
# ==========================================

ADAPTIVE_THRESHOLD = 80

WINDOW_LEVELS = {
    60: 30,
    30: 15
}

MAX_ITEMS = 100

print("Adaptive scraper siap.")
print(
    "Threshold :",
    ADAPTIVE_THRESHOLD
)

print(
    "Max items :",
    MAX_ITEMS
)

print(
    "Window levels:",
    WINDOW_LEVELS
)

Adaptive scraper siap.
Threshold : 80
Max items : 100
Window levels: {60: 30, 30: 15}


7. CHECKPOINT SYSTEM

In [36]:
# ============================================================
# CHECKPOINT SYSTEM
# ============================================================

CHECKPOINT_COLUMNS = [
    "checkpoint_key",
    "period",
    "keyword",
    "window_start_wib",
    "window_end_wib",
    "window_minutes",
    "status",
    "rows_raw",
    "rows_unique",
    "mock_count",
    "dataset_id",
    "search_query",
    "since_time",
    "until_time",
    "raw_file",
    "completed_at",
    "error"
]


def load_checkpoint():
    """
    Load checkpoint dan normalisasi struktur.
    """

    if not CHECKPOINT_FILE.exists():
        return pd.DataFrame(
            columns=CHECKPOINT_COLUMNS
        )

    checkpoint = pd.read_csv(
        CHECKPOINT_FILE
    )

    # Pastikan semua kolom tersedia
    for column in CHECKPOINT_COLUMNS:
        if column not in checkpoint.columns:
            checkpoint[column] = ""

    checkpoint = checkpoint[
        CHECKPOINT_COLUMNS
    ].copy()

    # Normalisasi string
    string_columns = [
        "checkpoint_key",
        "period",
        "keyword",
        "window_start_wib",
        "window_end_wib",
        "status",
        "dataset_id",
        "search_query",
        "since_time",
        "until_time",
        "raw_file",
        "completed_at",
        "error"
    ]

    for column in string_columns:
        checkpoint[column] = (
            checkpoint[column]
            .fillna("")
            .astype(str)
            .str.strip()
        )

    # Normalisasi numeric
    numeric_columns = [
        "window_minutes",
        "rows_raw",
        "rows_unique",
        "mock_count"
    ]

    for column in numeric_columns:
        checkpoint[column] = pd.to_numeric(
            checkpoint[column],
            errors="coerce"
        )

    # Rebuild checkpoint_key agar selalu konsisten
    checkpoint["checkpoint_key"] = (
        checkpoint["period"]
        + "|"
        + checkpoint["keyword"]
        + "|"
        + checkpoint["window_start_wib"]
        + "|"
        + checkpoint["window_end_wib"]
    )

    # Hapus row yang tidak punya identitas window
    valid_mask = (
        (checkpoint["period"] != "") &
        (checkpoint["keyword"] != "") &
        (checkpoint["window_start_wib"] != "") &
        (checkpoint["window_end_wib"] != "")
    )

    checkpoint = checkpoint.loc[
        valid_mask
    ].copy()

    # Hanya satu record per logical window
    checkpoint = (
        checkpoint
        .drop_duplicates(
            subset="checkpoint_key",
            keep="last"
        )
        .reset_index(drop=True)
    )

    return checkpoint


def save_checkpoint(checkpoint):

    checkpoint = checkpoint.copy()

    checkpoint.to_csv(
        CHECKPOINT_FILE,
        index=False,
        encoding="utf-8-sig"
    )


checkpoint = load_checkpoint()

print("Checkpoint rows :", len(checkpoint))
print("Checkpoint file :", CHECKPOINT_FILE)

Checkpoint rows : 1
Checkpoint file : C:\Project Mariana\data_bencana_nasional\checkpoint.csv


8. CHECK POINT KEY

In [37]:
# ============================================================
# CHECKPOINT HELPER
# ============================================================

COMPLETED_STATUSES = {
    "completed",
    "completed_min_window"
}

SPLIT_STATUS = "split"
FAILED_STATUS = "failed"


def checkpoint_key(
    period,
    keyword,
    start_wib,
    end_wib
):
    return (
        f"{str(period).strip()}|"
        f"{str(keyword).strip()}|"
        f"{str(start_wib).strip()}|"
        f"{str(end_wib).strip()}"
    )


def get_window_mask(
    checkpoint,
    period,
    keyword,
    start_wib,
    end_wib
):

    if checkpoint.empty:
        return pd.Series(
            False,
            index=checkpoint.index
        )

    key = checkpoint_key(
        period,
        keyword,
        start_wib,
        end_wib
    )

    return (
        checkpoint["checkpoint_key"]
        .astype(str)
        .str.strip()
        == key
    )


def get_window_status(
    checkpoint,
    period,
    keyword,
    start_wib,
    end_wib
):

    mask = get_window_mask(
        checkpoint,
        period,
        keyword,
        start_wib,
        end_wib
    )

    if not mask.any():
        return None

    matched = checkpoint.loc[
        mask
    ].copy()

    if matched.empty:
        return None

    # Karena checkpoint sudah unik,
    # hanya boleh ada satu record.
    if len(matched) > 1:
        raise RuntimeError(
            "Ditemukan duplicate logical checkpoint "
            f"untuk {period} | {keyword} | "
            f"{start_wib} → {end_wib}"
        )

    status = str(
        matched.iloc[0]["status"]
    ).strip()

    return status if status else None


def is_window_completed(
    checkpoint,
    period,
    keyword,
    start_wib,
    end_wib
):

    status = get_window_status(
        checkpoint,
        period,
        keyword,
        start_wib,
        end_wib
    )

    return status in COMPLETED_STATUSES

In [38]:
# ==========================================
# GET WINDOW STATUS
# ==========================================

def get_window_status(
    checkpoint,
    period,
    keyword,
    start_wib,
    end_wib
):
    """
    Mengambil status terbaru dari suatu window.

    Return:
    - None jika belum ada checkpoint
    - completed
    - completed_min_window
    - split
    - failed
    """

    if checkpoint.empty:
        return None

    mask = get_window_mask(
        checkpoint,
        period,
        keyword,
        start_wib,
        end_wib
    )

    if not mask.any():
        return None

    matched = checkpoint.loc[mask].copy()

    if matched.empty:
        return None

    # Karena update_checkpoint mengganti record lama,
    # biasanya hanya ada satu record.
    # Ambil record terakhir untuk keamanan.
    latest = matched.iloc[-1]

    status = str(
        latest.get("status", "")
    ).strip()

    if not status:
        return None

    return status


9. SCRAPE 1 WINDOW


In [39]:
# ==========================================
# SCRAPE SATU WINDOW
# ==========================================

def scrape_window_v3(
    search_query_base,
    start_wib,
    end_wib,
    max_items=100
):

    # =========================================================
    # 1. KONVERSI WIB → UTC
    # =========================================================

    start_dt_wib = datetime.strptime(
        start_wib,
        "%Y-%m-%d %H:%M:%S"
    )

    end_dt_wib = datetime.strptime(
        end_wib,
        "%Y-%m-%d %H:%M:%S"
    )

    start_dt_utc = (
        start_dt_wib
        - timedelta(hours=7)
    )

    end_dt_utc = (
        end_dt_wib
        - timedelta(hours=7)
    )

    start_utc = (
        start_dt_utc
        .strftime("%Y-%m-%d_%H:%M:%S_UTC")
    )

    end_utc = (
        end_dt_utc
        .strftime("%Y-%m-%d_%H:%M:%S_UTC")
    )


    # =========================================================
    # 2. SEARCH QUERY
    # =========================================================
    #
    # Actor baru menggunakan sintaks pencarian X/Twitter.
    #
    # lang:id = bahasa Indonesia
    # since   = batas awal
    # until   = batas akhir
    #
    # Semua filtering dimasukkan ke searchTerms.
    # =========================================================

    search_query = (
        f"{search_query_base} "
        f"lang:id "
        f"since:{start_utc} "
        f"until:{end_utc}"
    )


    # =========================================================
    # 3. INPUT ACTOR
    # =========================================================
    #
    # Jangan kirim field optional kosong seperti:
    # card_name=""
    # from=""
    # to=""
    # near=""
    # dll.
    #
    # Actor hanya menerima input yang benar-benar diperlukan.
    # =========================================================

    run_input = {
        "searchTerms": [
            search_query
        ],
        "maxItems": max_items,
        "queryType": "Latest"
    }


    # =========================================================
    # 4. PRINT QUERY
    # =========================================================

    print()
    print("=" * 80)
    print("APIFY SCRAPE WINDOW")
    print("=" * 80)

    print(
        "Actor ID   :",
        ACTOR_ID
    )

    print(
        "Start WIB  :",
        start_wib
    )

    print(
        "End WIB    :",
        end_wib
    )

    print()
    print("Search query:")
    print(search_query)

    print()
    print("Run input:")
    print(run_input)


    # =========================================================
    # 5. JALANKAN ACTOR
    # =========================================================

    run = client.actor(
        ACTOR_ID
    ).call(
        run_input=run_input
    )


    # =========================================================
    # 6. AMBIL DATASET ID
    # =========================================================

    if isinstance(run, dict):

        dataset_id = (
            run["defaultDatasetId"]
        )

    else:

        dataset_id = (
            run.default_dataset_id
        )


    # =========================================================
    # 7. AMBIL HASIL DATASET
    # =========================================================

    items = list(
        client.dataset(
            dataset_id
        ).iterate_items()
    )


    # =========================================================
    # 8. CONVERT KE DATAFRAME
    # =========================================================

    df_raw = pd.DataFrame(
        items
    )


    # =========================================================
    # 9. OUTPUT INFORMASI
    # =========================================================

    print()
    print("=" * 80)
    print("HASIL SCRAPING")
    print("=" * 80)

    print(
        "Dataset ID :",
        dataset_id
    )

    print(
        "Rows       :",
        len(df_raw)
    )

    print(
        "Columns    :",
        len(df_raw.columns)
    )

    print()
    print("Kolom:")
    print(
        df_raw.columns.tolist()
    )


    # =========================================================
    # 10. RETURN
    # =========================================================

    return {
        "df_raw": df_raw,
        "dataset_id": dataset_id,
        "search_query": search_query,
        "since_time": start_utc,
        "until_time": end_utc
    }

In [40]:
# ============================================================
# SMOKE TEST ACTOR BARU
# ============================================================

print()
print("=" * 80)
print("SMOKE TEST ACTOR BARU")
print("=" * 80)

print(
    "Actor ID :",
    ACTOR_ID
)


try:

    test_result = scrape_window_v3(
        search_query_base='"Palu"',
        start_wib="2018-09-28 18:00:00",
        end_wib="2018-09-28 19:00:00",
        max_items=20
    )


    df_test = test_result["df_raw"]


    print()
    print("=" * 80)
    print("SMOKE TEST BERHASIL")
    print("=" * 80)

    print(
        "Rows      :",
        len(df_test)
    )

    print(
        "Columns   :",
        df_test.columns.tolist()
    )

    print()
    print("5 DATA PERTAMA:")
    print(
        df_test.head(5).to_string()
    )


except Exception as e:

    print()
    print("=" * 80)
    print("SMOKE TEST GAGAL")
    print("=" * 80)

    print(
        "Error type :",
        type(e).__name__
    )

    print(
        "Error      :",
        str(e)
    )


SMOKE TEST ACTOR BARU
Actor ID : CJdippxWmn9uRfooo

APIFY SCRAPE WINDOW
Actor ID   : CJdippxWmn9uRfooo
Start WIB  : 2018-09-28 18:00:00
End WIB    : 2018-09-28 19:00:00

Search query:
"Palu" lang:id since:2018-09-28_11:00:00_UTC until:2018-09-28_12:00:00_UTC

Run input:
{'searchTerms': ['"Palu" lang:id since:2018-09-28_11:00:00_UTC until:2018-09-28_12:00:00_UTC'], 'maxItems': 20, 'queryType': 'Latest'}


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:nO8hF0761IhM1xCd1] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:nO8hF0761IhM1xCd1] -> 2026-10-10T08:00:41.775Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:nO8hF0761IhM1xCd1] -> 2026-10-10T08:00:41.779Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:nO8hF0761IhM1xCd1] -> 2026-10-10T08:00:41.864Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:nO8hF0761IhM1xCd1] -> 2026-10-10T08:00:41.866Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:nO8hF0761IhM1xCd1] -> 2026-10-10T08:00:49.111Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru


HASIL SCRAPING
Dataset ID : JFtDhdQ7wTgktC5Aj
Rows       : 15
Columns    : 3

Kolom:
['type', 'id', 'text']

SMOKE TEST BERHASIL
Rows      : 15
Columns   : ['type', 'id', 'text']

5 DATA PERTAMA:
         type  id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [41]:
# ============================================================
# INISIALISASI APIFY CLIENT
# ============================================================

import os
from apify_client import ApifyClient

# Ambil token dari environment variable
TOKEN = os.getenv("APIFY_API_TOKEN")

if not TOKEN:
    raise RuntimeError(
        "APIFY_API_TOKEN tidak ditemukan. "
        "Periksa konfigurasi token Apify."
    )

# Inisialisasi client
client = ApifyClient(TOKEN)

# Actor yang sedang diuji
ACTOR_ID = "CJdippxWmn9uRfooo"

print("Apify client berhasil diinisialisasi.")
print("Actor ID:", ACTOR_ID)
print("Token tersedia:", bool(TOKEN))

Apify client berhasil diinisialisasi.
Actor ID: CJdippxWmn9uRfooo
Token tersedia: True


In [42]:
# ============================================================
# DIAGNOSTIK ACTOR TANPA FILTER TANGGAL
# Memeriksa apakah hasilnya tweet asli atau mock_tweet
# ============================================================

import pandas as pd

print("=" * 70)
print("DIAGNOSTIK ACTOR TANPA FILTER TANGGAL")
print("=" * 70)

run_input_test = {
    "searchTerms": ['"Palu" lang:id'],
    "maxItems": 20,
    "queryType": "Latest"
}

try:
    # 1. Jalankan Actor
    run_test = client.actor(ACTOR_ID).call(
        run_input=run_input_test
    )

    # 2. Ambil dataset ID dari objek Run
    dataset_id_test = run_test.default_dataset_id

    if not dataset_id_test:
        raise RuntimeError(
            "Dataset ID tidak ditemukan pada hasil Actor."
        )

    print("Actor ID  :", ACTOR_ID)
    print("Dataset ID:", dataset_id_test)

    # 3. Ambil isi dataset
    items_test = list(
        client.dataset(dataset_id_test).iterate_items()
    )

    df_diagnostic = pd.DataFrame(items_test)

    print("\n" + "=" * 70)
    print("RINGKASAN HASIL")
    print("=" * 70)

    print("Jumlah baris:", len(df_diagnostic))
    print("Daftar kolom:", df_diagnostic.columns.tolist())

    # 4. Periksa tipe data
    if "type" in df_diagnostic.columns:
        print("\nDistribusi tipe data:")
        print(
            df_diagnostic["type"]
            .value_counts(dropna=False)
            .to_string()
        )

        genuine_count = int(
            (df_diagnostic["type"] != "mock_tweet").sum()
        )

        mock_count = int(
            (df_diagnostic["type"] == "mock_tweet").sum()
        )

        print("\nJumlah data non-mock:", genuine_count)
        print("Jumlah mock_tweet   :", mock_count)

    # 5. Perlihatkan sampel data
    print("\nLima baris pertama:")
    display(df_diagnostic.head(5))

    # 6. Beri peringatan jika dataset tidak memiliki hasil
    if df_diagnostic.empty:
        print("\nPERINGATAN: Dataset kosong.")

    elif (
        "type" in df_diagnostic.columns
        and (df_diagnostic["type"] == "mock_tweet").all()
    ):
        print(
            "\nPERINGATAN: Semua baris merupakan mock_tweet, "
            "bukan tweet asli."
        )

    else:
        print(
            "\nDataset berhasil dibaca. "
            "Periksa sampel sebelum digunakan untuk penelitian."
        )

except Exception as e:
    print("\nDIAGNOSTIK GAGAL")
    print("Jenis error:", type(e).__name__)
    print("Pesan      :", str(e))

DIAGNOSTIK ACTOR TANPA FILTER TANGGAL


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:aP61gBsSONjQe616N] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:aP61gBsSONjQe616N] -> 2026-10-10T08:01:40.806Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:aP61gBsSONjQe616N] -> 2026-10-10T08:01:40.810Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:aP61gBsSONjQe616N] -> 2026-10-10T08:01:40.891Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:aP61gBsSONjQe616N] -> 2026-10-10T08:01:40.908Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:aP61gBsSONjQe616N] -> 2026-10-10T08:01:47.374Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

Actor ID  : CJdippxWmn9uRfooo
Dataset ID: yMZwvikabOZFdXnbt

RINGKASAN HASIL
Jumlah baris: 40
Daftar kolom: ['type', 'id', 'url', 'twitterUrl', 'text', 'source', 'retweetCount', 'replyCount', 'likeCount', 'quoteCount', 'viewCount', 'createdAt', 'lang', 'bookmarkCount', 'isReply', 'inReplyToId', 'conversationId', 'inReplyToUserId', 'inReplyToUsername', 'isPinned', 'author', 'extendedEntities', 'card', 'place', 'entities', 'reply_to_user_results', 'quoted_tweet_results', 'quoted_tweet', 'retweeted_tweet', 'isConversationControlled', 'searchTermIndex']

Distribusi tipe data:
type
tweet    40

Jumlah data non-mock: 40
Jumlah mock_tweet   : 0

Lima baris pertama:


,type,id,url,twitterUrl,text,source,retweetCount,replyCount,likeCount,quoteCount,...,extendedEntities,card,place,entities,reply_to_user_results,quoted_tweet_results,quoted_tweet,retweeted_tweet,isConversationControlled,searchTermIndex
0,tweet,432090561793646592,https://x.com/MilanistiOrId/status/43209056179...,https://twitter.com/MilanistiOrId/status/43209...,"[PALU] Napoli vs #ACMilan | Minggu,09-02-14 pk...",,4,0,0,0,...,{},None,{},"{'hashtags': [{'indices': [17, 25], 'text': 'A...",None,None,None,None,False,0
1,tweet,431469762359472128,https://x.com/nulisbuku/status/431469762359472128,https://twitter.com/nulisbuku/status/431469762...,"""@retnogaluh: Rentan waktu yang enggan berlalu...",,2,1,0,0,...,{},None,{},"{'hashtags': [{'indices': [121, 132], 'text': ...",None,None,None,None,False,0
2,tweet,431430088312123392,https://x.com/liputan6dotcom/status/4314300883...,https://twitter.com/liputan6dotcom/status/4314...,2 Jenazah Korban Baku Tembak di Poso Akan Dibe...,,3,0,3,0,...,{},None,{},"{'urls': [{'display_url': 'lp6.co/214M', 'expa...",None,None,None,None,False,0
3,tweet,431302383486775296,https://x.com/Aradrizki/status/431302383486775296,https://twitter.com/Aradrizki/status/431302383...,Do'ain aja biar secepatnya bsa ksna :) RT @Dhy...,,12,6,5,0,...,{},None,{},"{'user_mentions': [{'id_str': '329684631', 'in...",None,None,None,None,False,0
4,tweet,431231161592209408,https://x.com/AmazingMentor/status/43123116159...,https://twitter.com/AmazingMentor/status/43123...,jika satu-satunya nya alat yang kamu pnya adal...,,18,0,1,0,...,{},None,{},{},None,None,None,None,False,0



Dataset berhasil dibaca. Periksa sampel sebelum digunakan untuk penelitian.


In [43]:
# ============================================================
# DIAGNOSTIK PEMBANDING: HISTORIS VS MODERN
# ============================================================

from datetime import datetime, timezone
import pandas as pd

def to_unix_utc(dt):
    return int(dt.timestamp())

# Rentang modern — kontrol
recent_start = datetime(
    2026, 10, 9, 16, 0, 0,
    tzinfo=timezone.utc
)
recent_end = datetime(
    2026, 10, 9, 17, 0, 0,
    tzinfo=timezone.utc
)

# Rentang historis — penelitian Palu
historic_start = datetime(
    2018, 9, 28, 10, 0, 0,
    tzinfo=timezone.utc
)
historic_end = datetime(
    2018, 9, 28, 11, 0, 0,
    tzinfo=timezone.utc
)

queries = [
    (
        "KONTROL_MODERN",
        f"Palu since_time:{to_unix_utc(recent_start)} "
        f"until_time:{to_unix_utc(recent_end)}"
    ),
    (
        "PALU_HISTORIS",
        f"Palu since_time:{to_unix_utc(historic_start)} "
        f"until_time:{to_unix_utc(historic_end)}"
    )
]

run_input = {
    "searchTerms": [q for _, q in queries],
    "maxItems": 20,
    "queryType": "Latest"
}

print("QUERY YANG DIUJI")
for label, query in queries:
    print(f"\n{label}: {query}")

try:
    run = client.actor(ACTOR_ID).call(
        run_input=run_input
    )

    dataset_id = run.default_dataset_id

    items = list(
        client.dataset(dataset_id).iterate_items()
    )

    df = pd.DataFrame(items)

    print("\nDATASET ID:", dataset_id)
    print("TOTAL BARIS:", len(df))
    print("KOLOM:", df.columns.tolist())

    if df.empty:
        print("Dataset kosong.")

    elif "searchTermIndex" in df.columns:
        df["searchTermIndex"] = pd.to_numeric(
            df["searchTermIndex"],
            errors="coerce"
        )

        for idx, (label, query) in enumerate(queries):
            part = df[
                df["searchTermIndex"] == idx
            ].copy()

            print("\n" + "=" * 60)
            print(label)
            print("=" * 60)
            print("Jumlah baris:", len(part))

            if "type" in part.columns:
                print(
                    part["type"]
                    .value_counts(dropna=False)
                    .to_string()
                )

            if "createdAt" in part.columns:
                print("\nWaktu publikasi sampel:")
                display(
                    part[
                        [
                            c for c in
                            ["type", "id", "createdAt", "text"]
                            if c in part.columns
                        ]
                    ].head(5)
                )
    else:
        print("\nDistribusi tipe:")
        if "type" in df.columns:
            print(df["type"].value_counts(dropna=False))

        print("\nSampel:")
        display(df.head(10))

except Exception as e:
    print("DIAGNOSTIK GAGAL")
    print(type(e).__name__, str(e))

QUERY YANG DIUJI

KONTROL_MODERN: Palu since_time:1791561600 until_time:1791565200

PALU_HISTORIS: Palu since_time:1538128800 until_time:1538132400


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:y45PIcTe9q1deFp2I] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:y45PIcTe9q1deFp2I] -> 2026-10-10T08:02:23.641Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:y45PIcTe9q1deFp2I] -> 2026-10-10T08:02:23.643Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:y45PIcTe9q1deFp2I] -> 2026-10-10T08:02:23.715Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:y45PIcTe9q1deFp2I] -> 2026-10-10T08:02:23.716Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:y45PIcTe9q1deFp2I] -> 2026-10-10T08:02:30.218Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru


DATASET ID: oUaTuKCcXB5002Lr5
TOTAL BARIS: 54
KOLOM: ['type', 'id', 'url', 'twitterUrl', 'text', 'source', 'retweetCount', 'replyCount', 'likeCount', 'quoteCount', 'viewCount', 'createdAt', 'lang', 'bookmarkCount', 'isReply', 'inReplyToId', 'conversationId', 'inReplyToUserId', 'inReplyToUsername', 'isPinned', 'author', 'extendedEntities', 'card', 'place', 'entities', 'reply_to_user_results', 'quoted_tweet_results', 'quoted_tweet', 'retweeted_tweet', 'isConversationControlled', 'searchTermIndex', 'isQuote']

KONTROL_MODERN
Jumlah baris: 14
type
tweet    14

Waktu publikasi sampel:


,type,id,createdAt,text
0,tweet,2108599470401200134,Fri Oct 09 16:44:24 +0000 2026,un día más en la tierra para conseguir que vea...
1,tweet,2108599259096633443,Fri Oct 09 16:43:33 +0000 2026,Akhirnya kepalanya aku potong pake pisau dan p...
2,tweet,2108599073695522958,Fri Oct 09 16:42:49 +0000 2026,My palu kinda clean https://t.co/Y5H270DpYU
3,tweet,2108596177684091169,Fri Oct 09 16:31:19 +0000 2026,BENERAN PACARAN FIKS. PUKUL PALU
4,tweet,2108592416752361689,Fri Oct 09 16:16:22 +0000 2026,cosas increíbles sucediendo en mi portátil htt...



PALU_HISTORIS
Jumlah baris: 40
type
tweet    40

Waktu publikasi sampel:


,type,id,createdAt,text
14,tweet,1045629007578812416,Fri Sep 28 10:59:16 +0000 2018,【地震速報 M5.4】9月28日 19:39頃、インドネシアのPaluから北24kmの地点で...
15,tweet,1045628963782086656,Fri Sep 28 10:59:06 +0000 2018,"Indonesia, terremoto di magnitudo 7.5 a 80 km ..."
16,tweet,1045628904713711621,Fri Sep 28 10:58:52 +0000 2018,New #Earthquake. 🌎\n5.6 (Mb) - 30 km al norest...
17,tweet,1045628897356738560,Fri Sep 28 10:58:50 +0000 2018,#Palu #PLWairport / #WAFFairport has damage ca...
18,tweet,1045628602803507201,Fri Sep 28 10:57:40 +0000 2018,Noticias\nTerremoto de magnitud 7.5Mw a 79 km ...


In [44]:
# ============================================================
# DIAGNOSTIK 4 — VALIDASI FILTER BAHASA
# Membandingkan semua bahasa vs bahasa Indonesia
# ============================================================

import pandas as pd

since_unix = 1538128800  # 28 Sep 2018, 10:00 UTC
until_unix = 1538132400  # 28 Sep 2018, 11:00 UTC

queries = [
    (
        "TANPA_FILTER_BAHASA",
        f"Palu since_time:{since_unix} until_time:{until_unix}"
    ),
    (
        "BAHASA_INDONESIA",
        f"Palu lang:in since_time:{since_unix} until_time:{until_unix}"
    )
]

print("=" * 70)
print("DIAGNOSTIK FILTER BAHASA")
print("=" * 70)

for label, query in queries:
    print(f"{label}: {query}")

try:
    run = client.actor(ACTOR_ID).call(
        run_input={
            "searchTerms": [q for _, q in queries],
            "maxItems": 20,
            "queryType": "Latest"
        }
    )

    items = list(
        client.dataset(
            run.default_dataset_id
        ).iterate_items()
    )

    df_lang_test = pd.DataFrame(items)

    print("\nTotal baris:", len(df_lang_test))
    print("Kolom:", df_lang_test.columns.tolist())

    if "searchTermIndex" not in df_lang_test.columns:
        print(
            "\nKolom searchTermIndex tidak ditemukan. "
            "Hasil tidak dapat dipisahkan dengan aman per query."
        )
        display(df_lang_test.head(10))

    else:
        df_lang_test["searchTermIndex"] = pd.to_numeric(
            df_lang_test["searchTermIndex"],
            errors="coerce"
        )

        for index, (label, query) in enumerate(queries):
            part = df_lang_test[
                df_lang_test["searchTermIndex"] == index
            ].copy()

            print("\n" + "-" * 70)
            print(label)
            print("-" * 70)
            print("Jumlah baris:", len(part))

            if "type" in part.columns:
                print("\nTipe data:")
                print(part["type"].value_counts(dropna=False))

            real_tweets = part[
                part["type"].eq("tweet")
            ].copy() if "type" in part.columns else part.iloc[0:0]

            if "lang" in real_tweets.columns:
                print("\nDistribusi bahasa tweet asli:")
                print(
                    real_tweets["lang"]
                    .value_counts(dropna=False)
                    .to_string()
                )

            if "createdAt" in real_tweets.columns:
                real_tweets["createdAt_utc"] = pd.to_datetime(
                    real_tweets["createdAt"],
                    format="%a %b %d %H:%M:%S %z %Y",
                    utc=True,
                    errors="coerce"
                )

                start = pd.Timestamp(
                    "2018-09-28 10:00:00",
                    tz="UTC"
                )
                end = pd.Timestamp(
                    "2018-09-28 11:00:00",
                    tz="UTC"
                )

                valid = real_tweets[
                    real_tweets["createdAt_utc"].ge(start)
                    & real_tweets["createdAt_utc"].lt(end)
                ]

                print(
                    "\nTweet asli dalam rentang waktu:",
                    len(valid),
                    "/",
                    len(real_tweets)
                )

                display(
                    valid[
                        [
                            c for c in
                            ["id", "createdAt", "lang", "text"]
                            if c in valid.columns
                        ]
                    ].head(5)
                )

except Exception as e:
    print("\nDIAGNOSTIK GAGAL")
    print("Jenis error:", type(e).__name__)
    print("Pesan:", str(e))

DIAGNOSTIK FILTER BAHASA
TANPA_FILTER_BAHASA: Palu since_time:1538128800 until_time:1538132400
BAHASA_INDONESIA: Palu lang:in since_time:1538128800 until_time:1538132400


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:iapVhheH2hxsjPbbb] -> 2026-10-10T08:03:43.690Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:iapVhheH2hxsjPbbb] -> 2026-10-10T08:03:43.693Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:iapVhheH2hxsjPbbb] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:iapVhheH2hxsjPbbb] -> 2026-10-10T08:03:43.757Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:iapVhheH2hxsjPbbb] -> 2026-10-10T08:03:43.758Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:iapVhheH2hxsjPbbb] -> 2026-10-10T08:03:50.280Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru


Total baris: 80
Kolom: ['type', 'id', 'url', 'twitterUrl', 'text', 'source', 'retweetCount', 'replyCount', 'likeCount', 'quoteCount', 'viewCount', 'createdAt', 'lang', 'bookmarkCount', 'isReply', 'inReplyToId', 'conversationId', 'inReplyToUserId', 'inReplyToUsername', 'isPinned', 'author', 'extendedEntities', 'card', 'place', 'entities', 'reply_to_user_results', 'quoted_tweet_results', 'quoted_tweet', 'retweeted_tweet', 'isConversationControlled', 'searchTermIndex']

----------------------------------------------------------------------
TANPA_FILTER_BAHASA
----------------------------------------------------------------------
Jumlah baris: 40

Tipe data:
type
tweet    40
Name: count, dtype: int64

Distribusi bahasa tweet asli:
lang
in    8
en    8
ja    6
es    4
id    4
it    2
ro    2
tr    2
ca    2
fr    2

Tweet asli dalam rentang waktu: 40 / 40


,id,createdAt,lang,text
0,1045629007578812416,Fri Sep 28 10:59:16 +0000 2018,ja,【地震速報 M5.4】9月28日 19:39頃、インドネシアのPaluから北24kmの地点で...
1,1045628963782086656,Fri Sep 28 10:59:06 +0000 2018,it,"Indonesia, terremoto di magnitudo 7.5 a 80 km ..."
2,1045628904713711621,Fri Sep 28 10:58:52 +0000 2018,in,New #Earthquake. 🌎\n5.6 (Mb) - 30 km al norest...
3,1045628897356738560,Fri Sep 28 10:58:50 +0000 2018,en,#Palu #PLWairport / #WAFFairport has damage ca...
4,1045628602803507201,Fri Sep 28 10:57:40 +0000 2018,es,Noticias\nTerremoto de magnitud 7.5Mw a 79 km ...



----------------------------------------------------------------------
BAHASA_INDONESIA
----------------------------------------------------------------------
Jumlah baris: 40

Tipe data:
type
tweet    40
Name: count, dtype: int64

Distribusi bahasa tweet asli:
lang
id    25
in    15

Tweet asli dalam rentang waktu: 40 / 40


,id,createdAt,lang,text
40,1045628904713711621,Fri Sep 28 10:58:52 +0000 2018,in,New #Earthquake. 🌎\n5.6 (Mb) - 30 km al norest...
41,1045628314889539584,Fri Sep 28 10:56:31 +0000 2018,in,Video warga di Palu pasca gempa. Jalanan retak...
42,1045628015047192577,Fri Sep 28 10:55:20 +0000 2018,in,"Mall Tatura Palu rusak parah akibat gempa 7,7 ..."
43,1045627822792830976,Fri Sep 28 10:54:34 +0000 2018,id,Masyarakat Kota Palu Berhamburan Keluar Rumah ...
44,1045627294243422208,Fri Sep 28 10:52:28 +0000 2018,id,Innalillahi wa innailaihi rooji'uun. Ya Allah ...


In [24]:
# DIAGNOSTIK FINAL: VALIDASI FILTER WAKTU

import pandas as pd

# Gunakan fungsi scraping yang sudah terbukti
# menghasilkan tweet asli pada pengujianmu.
test_result = scrape_window_v3(
    search_query_base="Palu",
    start_wib="2018-09-28 17:00:00",
    end_wib="2018-09-28 18:00:00",
    max_items=20
)

df_test = test_result["df_raw"].copy()

print("=" * 70)
print("VALIDASI FILTER WAKTU")
print("=" * 70)
print("Jumlah data:", len(df_test))
print("Kolom tersedia:", df_test.columns.tolist())

if "createdAt" in df_test.columns:
    df_test["createdAt_utc"] = pd.to_datetime(
        df_test["createdAt"],
        utc=True,
        errors="coerce"
    )

    start_utc = pd.Timestamp("2018-09-28 10:00:00", tz="UTC")
    end_utc = pd.Timestamp("2018-09-28 11:00:00", tz="UTC")

    valid_time = (
        df_test["createdAt_utc"].ge(start_utc)
        & df_test["createdAt_utc"].lt(end_utc)
    )

    print("Tweet dengan waktu valid:", int(valid_time.sum()))
    print("Tweet di luar rentang:", int((~valid_time).sum()))
    print("\nSampel tanggal publikasi:")
    print(
        df_test[
            ["id", "createdAt", "createdAt_utc", "text"]
        ].head(10).to_string(index=False)
    )
else:
    print("Kolom createdAt tidak ditemukan.")


APIFY SCRAPE WINDOW
Actor ID   : CJdippxWmn9uRfooo
Start WIB  : 2018-09-28 17:00:00
End WIB    : 2018-09-28 18:00:00

Search query:
Palu lang:id since:2018-09-28_10:00:00_UTC until:2018-09-28_11:00:00_UTC

Run input:
{'searchTerms': ['Palu lang:id since:2018-09-28_10:00:00_UTC until:2018-09-28_11:00:00_UTC'], 'maxItems': 20, 'queryType': 'Latest'}


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:98zktybEm4ZNp0ZqO] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:98zktybEm4ZNp0ZqO] -> 2026-10-10T07:54:49.168Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:98zktybEm4ZNp0ZqO] -> 2026-10-10T07:54:49.171Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:98zktybEm4ZNp0ZqO] -> 2026-10-10T07:54:49.267Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:98zktybEm4ZNp0ZqO] -> 2026-10-10T07:54:49.268Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:98zktybEm4ZNp0ZqO] -> 2026-10-10T07:54:56.279Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru


HASIL SCRAPING
Dataset ID : TdL4UR1V8fI0ZQuLY
Rows       : 15
Columns    : 3

Kolom:
['type', 'id', 'text']
VALIDASI FILTER WAKTU
Jumlah data: 15
Kolom tersedia: ['type', 'id', 'text']
Kolom createdAt tidak ditemukan.


In [46]:
# Periksa nama variabel yang tersedia tanpa menampilkan isi token
[name for name in globals() if "token" in name.lower() or "apify" in name.lower()]

['ApifyClient', 'TOKEN']

In [48]:
from apify_client import ApifyClient
import pandas as pd

# Inisialisasi Apify Client
client = ApifyClient(TOKEN)

# Actor ID
actor_id = "CJdippxWmn9uRfooo"

# Jalankan Actor
run = client.actor(actor_id).call(
    run_input={
        "searchTerms": ["Palu lang:in"],
        "maxItems": 10,
        "queryType": "Latest"
    }
)

# Konversi objek Run menjadi dictionary jika diperlukan
if isinstance(run, dict):
    run_data = run
elif hasattr(run, "model_dump"):
    run_data = run.model_dump(by_alias=True)
elif hasattr(run, "to_dict"):
    run_data = run.to_dict()
else:
    raise TypeError(
        f"Format Run belum dikenali: {type(run)}"
    )

# Ambil informasi dataset
dataset_id = (
    run_data.get("defaultDatasetId")
    or run_data.get("default_dataset_id")
)

if not dataset_id:
    raise ValueError(
        f"Dataset ID tidak ditemukan. Field tersedia: {list(run_data.keys())}"
    )

# Ambil hasil scraping
df_test = pd.DataFrame(
    client.dataset(dataset_id).iterate_items()
)

# Tampilkan hasil
print("=" * 70)
print("HASIL PENGUJIAN ACTOR APIFY")
print("=" * 70)

print("Status run :", run_data.get("status"))
print("Dataset ID :", dataset_id)
print("Jumlah data:", len(df_test))
print("Kolom      :", df_test.columns.tolist())

if not df_test.empty:
    print("\nJenis data:")

    if "type" in df_test.columns:
        print(df_test["type"].value_counts(dropna=False))

    print("\nContoh hasil:")

    kolom_tampil = [
        col for col in ["id", "text", "createdAt", "lang"]
        if col in df_test.columns
    ]

    print(df_test[kolom_tampil].head(5).to_string(index=False))

    print("\nApakah ada kolom createdAt?", "createdAt" in df_test.columns)

    # Validasi apakah data tweet asli
    if (
        "createdAt" in df_test.columns
        and "type" in df_test.columns
        and (df_test["type"] == "tweet").any()
    ):
        print("\nVALIDASI: Tweet asli terdeteksi.")
    else:
        print("\nVALIDASI: Data belum memenuhi struktur tweet asli.")
else:
    print("\nDataset kosong.")

[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:teLlzDAhvnFKDPlwQ] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:teLlzDAhvnFKDPlwQ] -> 2026-10-10T08:07:26.711Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:teLlzDAhvnFKDPlwQ] -> 2026-10-10T08:07:26.714Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:teLlzDAhvnFKDPlwQ] -> 2026-10-10T08:07:26.804Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:teLlzDAhvnFKDPlwQ] -> 2026-10-10T08:07:26.814Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:teLlzDAhvnFKDPlwQ] -> 2026-10-10T08:07:32.930Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

HASIL PENGUJIAN ACTOR APIFY
Status run : SUCCEEDED
Dataset ID : 3iClGdyGz78bWQFgK
Jumlah data: 40
Kolom      : ['type', 'id', 'url', 'twitterUrl', 'text', 'source', 'retweetCount', 'replyCount', 'likeCount', 'quoteCount', 'viewCount', 'createdAt', 'lang', 'bookmarkCount', 'isReply', 'inReplyToId', 'conversationId', 'inReplyToUserId', 'inReplyToUsername', 'isPinned', 'author', 'extendedEntities', 'card', 'place', 'entities', 'reply_to_user_results', 'quoted_tweet_results', 'quoted_tweet', 'retweeted_tweet', 'isConversationControlled', 'searchTermIndex', 'isQuote']

Jenis data:
type
tweet    40
Name: count, dtype: int64

Contoh hasil:
                 id                                                                                                                                                                                                                                                                                            text                      createdAt lang
210883097076184

In [49]:
import pandas as pd

actor_id = "CJdippxWmn9uRfooo"

queries = {
    "FORMAT_TANGGAL": (
        "gempa Palu lang:in "
        "since:2018-09-28 until:2018-09-30"
    ),
    "FORMAT_UNIX": (
        "gempa Palu lang:in "
        "since_time:1538128800 until_time:1538301600"
    )
}

hasil = []

for nama, query in queries.items():
    print("\n" + "=" * 65)
    print("PENGUJIAN:", nama)
    print("QUERY:", query)
    print("=" * 65)

    run = client.actor(actor_id).call(
        run_input={
            "searchTerms": [query],
            "maxItems": 10,
            "queryType": "Latest"
        }
    )

    if isinstance(run, dict):
        run_data = run
    elif hasattr(run, "model_dump"):
        run_data = run.model_dump(by_alias=True)
    else:
        raise TypeError(f"Format Run tidak dikenali: {type(run)}")

    dataset_id = (
        run_data.get("defaultDatasetId")
        or run_data.get("default_dataset_id")
    )

    df = pd.DataFrame(
        client.dataset(dataset_id).iterate_items()
    )

    if "createdAt" in df.columns:
        tanggal = pd.to_datetime(
            df["createdAt"],
            format="%a %b %d %H:%M:%S %z %Y",
            utc=True,
            errors="coerce"
        )

        awal = pd.Timestamp("2018-09-28", tz="UTC")
        akhir = pd.Timestamp("2018-09-30", tz="UTC")

        valid = tanggal.ge(awal) & tanggal.lt(akhir)
        jumlah_valid = int(valid.sum())
    else:
        jumlah_valid = 0

    print("Status:", run_data.get("status"))
    print("Jumlah baris:", len(df))
    print("Kolom createdAt:", "createdAt" in df.columns)
    print("Tweet dalam rentang:", jumlah_valid)

    if "type" in df.columns:
        print("Jenis data:")
        print(df["type"].value_counts(dropna=False))

    if "createdAt" in df.columns:
        print("\nSampel tanggal:")
        print(df[["createdAt", "text"]].head(3).to_string(index=False))

    hasil.append({
        "metode": nama,
        "jumlah_baris": len(df),
        "tweet_valid_waktu": jumlah_valid,
        "ada_createdAt": "createdAt" in df.columns
    })

print("\n" + "=" * 65)
print("RINGKASAN PENGUJIAN")
print("=" * 65)

display(pd.DataFrame(hasil))


PENGUJIAN: FORMAT_TANGGAL
QUERY: gempa Palu lang:in since:2018-09-28 until:2018-09-30


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Y5KB503vM0cRP7CCJ] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Y5KB503vM0cRP7CCJ] -> 2026-10-10T08:10:34.125Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Y5KB503vM0cRP7CCJ] -> 2026-10-10T08:10:34.128Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Y5KB503vM0cRP7CCJ] -> 2026-10-10T08:10:34.215Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Y5KB503vM0cRP7CCJ] -> 2026-10-10T08:10:34.219Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Y5KB503vM0cRP7CCJ] -> 2026-10-10T08:10:40.280Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

Status: SUCCEEDED
Jumlah baris: 40
Kolom createdAt: True
Tweet dalam rentang: 0
Jenis data:
type
tweet    40
Name: count, dtype: int64

Sampel tanggal:
                     createdAt                                                                                                                                                                                                                                                                                                       text
Mon Oct 01 03:59:51 +0000 2018                                                                                                                                                                                                                  Gempa Palu, Sri Mulyani: 13 Jajaran Dirjen Pajak Belum Ada Kabar\nhttps://t.co/9WcDscHIWg
Mon Oct 01 03:59:20 +0000 2018 Selamat hari kopi sedunia. Jangan lupa berbagi kenikmatan dengan orang2 yg membutuhkan bantuan kita, terutama masyarakat korban gempa #Donggala dan #Palu ser

[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:aCQemf6OhMaiVdMr4] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:aCQemf6OhMaiVdMr4] -> 2026-10-10T08:11:05.812Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:aCQemf6OhMaiVdMr4] -> 2026-10-10T08:11:05.815Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:aCQemf6OhMaiVdMr4] -> 2026-10-10T08:11:05.887Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:aCQemf6OhMaiVdMr4] -> 2026-10-10T08:11:05.889Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:aCQemf6OhMaiVdMr4] -> 2026-10-10T08:11:12.179Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

Status: SUCCEEDED
Jumlah baris: 40
Kolom createdAt: True
Tweet dalam rentang: 0
Jenis data:
type
tweet    40
Name: count, dtype: int64

Sampel tanggal:
                     createdAt                                                                                                                                                                                                                                                                                                        text
Sun Sep 30 09:59:53 +0000 2018                                                                                                                                                                                        IMM Papua akan Galang Dana Peduli Bencana Palu-Donggala https://t.co/xsbvWW7TzN #Papua #peristiwa #galangdana #gempa
Sun Sep 30 09:59:52 +0000 2018 Kemarin, pagi-pagi, saya diberitahu bahwa Bu Cahaya di Kota Palu telah meninggal. Karena gempa. Dua orang yang menyampaikan kabar itu ke saya. Tapi, saya b

,metode,jumlah_baris,tweet_valid_waktu,ada_createdAt
0,FORMAT_TANGGAL,40,0,True
1,FORMAT_UNIX,40,0,True


In [50]:
import pandas as pd

# Run ID dari dua pengujian sebelumnya
RUN_IDS = {
    "FORMAT_TANGGAL": "Y5KB503vM0cRP7CCJ",
    "FORMAT_UNIX": "aCQemf6OhMaiVdMr4"
}

# Rentang waktu yang benar untuk masing-masing query
RENTANG = {
    "FORMAT_TANGGAL": (
        pd.Timestamp("2018-09-28 00:00:00", tz="UTC"),
        pd.Timestamp("2018-09-30 00:00:00", tz="UTC")
    ),
    "FORMAT_UNIX": (
        pd.to_datetime(1538128800, unit="s", utc=True),
        pd.to_datetime(1538301600, unit="s", utc=True)
    )
}

ringkasan = []

for metode, run_id in RUN_IDS.items():

    print("\n" + "=" * 70)
    print("VALIDASI:", metode)
    print("=" * 70)

    # Ambil hasil run lama, tanpa menjalankan Actor kembali
    run_lama = client.run(run_id).get()

    if run_lama is None:
        print("Run tidak ditemukan.")
        continue

    dataset_id = (
        run_lama.get("defaultDatasetId")
        if isinstance(run_lama, dict)
        else run_lama.default_dataset_id
    )

    df = pd.DataFrame(
        client.dataset(dataset_id).iterate_items()
    )

    if "createdAt" not in df.columns:
        print("Tidak ada kolom createdAt.")
        continue

    # Konversi waktu Twitter menjadi UTC
    df["tanggal_utc"] = pd.to_datetime(
        df["createdAt"],
        format="%a %b %d %H:%M:%S %z %Y",
        utc=True,
        errors="coerce"
    )

    awal, akhir = RENTANG[metode]

    valid_time = (
        df["tanggal_utc"].ge(awal)
        & df["tanggal_utc"].lt(akhir)
    )

    valid_tweet = (
        df["type"].eq("tweet")
        if "type" in df.columns
        else pd.Series(False, index=df.index)
    )

    valid = valid_time & valid_tweet

    jumlah_valid = int(valid.sum())
    total = len(df)

    print("Dataset ID       :", dataset_id)
    print("Total data       :", total)
    print("Awal UTC         :", awal)
    print("Akhir UTC        :", akhir)
    print("Tanggal minimum  :", df["tanggal_utc"].min())
    print("Tanggal maksimum :", df["tanggal_utc"].max())
    print("Gagal parse      :", int(df["tanggal_utc"].isna().sum()))
    print("Tweet valid      :", jumlah_valid)
    print("Tweet tidak valid:", total - jumlah_valid)

    if total > 0:
        print("Persentase valid :", f"{jumlah_valid / total * 100:.2f}%")

    print("\nSampel tweet yang VALID:")

    kolom = ["createdAt", "text"]

    if valid.any():
        print(
            df.loc[valid, kolom]
            .head(3)
            .to_string(index=False, max_colwidth=100)
        )
    else:
        print("Tidak ditemukan.")

    ringkasan.append({
        "metode": metode,
        "total_tweet": total,
        "tweet_valid": jumlah_valid,
        "tweet_tidak_valid": total - jumlah_valid,
        "persentase_valid": (
            round(jumlah_valid / total * 100, 2)
            if total else 0
        )
    })

print("\n" + "=" * 70)
print("RINGKASAN VALIDASI FINAL")
print("=" * 70)

display(pd.DataFrame(ringkasan))


VALIDASI: FORMAT_TANGGAL
Dataset ID       : kPtjxnyIvcS8dCDvJ
Total data       : 40
Awal UTC         : 2018-09-28 00:00:00+00:00
Akhir UTC        : 2018-09-30 00:00:00+00:00
Tanggal minimum  : 2018-10-01 03:53:15+00:00
Tanggal maksimum : 2018-10-01 03:59:51+00:00
Gagal parse      : 0
Tweet valid      : 0
Tweet tidak valid: 40
Persentase valid : 0.00%

Sampel tweet yang VALID:
Tidak ditemukan.

VALIDASI: FORMAT_UNIX
Dataset ID       : R939XGeHn0ZVECrfL
Total data       : 40
Awal UTC         : 2018-09-28 10:00:00+00:00
Akhir UTC        : 2018-09-30 10:00:00+00:00
Tanggal minimum  : 2018-09-30 09:50:25+00:00
Tanggal maksimum : 2018-09-30 09:59:54+00:00
Gagal parse      : 0
Tweet valid      : 40
Tweet tidak valid: 0
Persentase valid : 100.00%

Sampel tweet yang VALID:
                     createdAt                                                                                                 text
Sun Sep 30 09:59:53 +0000 2018 IMM Papua akan Galang Dana Peduli Bencana Palu-Donggala https

,metode,total_tweet,tweet_valid,tweet_tidak_valid,persentase_valid
0,FORMAT_TANGGAL,40,0,40,0.0
1,FORMAT_UNIX,40,40,0,100.0


In [51]:
from pathlib import Path
import pandas as pd
import json

OUTPUT = Path("data_raw_penelitian")
OUTPUT.mkdir(exist_ok=True)

DATASETS = {
    "PALU_FORMAT_TANGGAL": {
        "dataset_id": "kPtjxnyIvcS8dCDvJ",
        "start": "2018-09-28 00:00:00+00:00",
        "end": "2018-09-30 00:00:00+00:00"
    },
    "PALU_FORMAT_UNIX": {
        "dataset_id": "R939XGeHn0ZVECrfL",
        "start": "2018-09-28 10:00:00+00:00",
        "end": "2018-09-30 10:00:00+00:00"
    }
}

for nama, info in DATASETS.items():
    items = list(
        client.dataset(info["dataset_id"]).iterate_items()
    )

    df = pd.DataFrame(items)

    # Simpan semua data asli, termasuk yang tidak valid.
    with open(
        OUTPUT / f"{nama}_raw.jsonl",
        "w",
        encoding="utf-8"
    ) as f:
        for item in items:
            f.write(
                json.dumps(item, ensure_ascii=False, default=str)
                + "\n"
            )

    df.to_csv(
        OUTPUT / f"{nama}_raw.csv",
        index=False,
        encoding="utf-8-sig"
    )

    if df.empty or "createdAt" not in df.columns:
        print(nama, "| Tidak ditemukan tweet bertanggal.")
        continue

    df["createdAt_utc"] = pd.to_datetime(
        df["createdAt"],
        format="%a %b %d %H:%M:%S %z %Y",
        utc=True,
        errors="coerce"
    )

    awal = pd.Timestamp(info["start"])
    akhir = pd.Timestamp(info["end"])

    mask = (
        df["createdAt_utc"].ge(awal)
        & df["createdAt_utc"].lt(akhir)
        & df["type"].eq("tweet")
    )

    df_valid = df.loc[mask].copy()

    df_valid.to_csv(
        OUTPUT / f"{nama}_valid.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("=" * 65)
    print("Dataset:", nama)
    print("Total raw:", len(df))
    print("Tweet valid:", len(df_valid))
    print("Tanggal minimum:", df["createdAt_utc"].min())
    print("Tanggal maksimum:", df["createdAt_utc"].max())

print("\nSELESAI. Folder output:", OUTPUT.resolve())

Dataset: PALU_FORMAT_TANGGAL
Total raw: 40
Tweet valid: 0
Tanggal minimum: 2018-10-01 03:53:15+00:00
Tanggal maksimum: 2018-10-01 03:59:51+00:00
Dataset: PALU_FORMAT_UNIX
Total raw: 40
Tweet valid: 40
Tanggal minimum: 2018-09-30 09:50:25+00:00
Tanggal maksimum: 2018-09-30 09:59:54+00:00

SELESAI. Folder output: C:\Project Mariana\LSTM BENCANA\01_Raw Data\data_raw_penelitian


In [ ]:
import json
import pandas as pd

from pathlib import Path
from itertools import zip_longest
from datetime import timedelta

# =====================================================
# KONFIGURASI
# =====================================================

ACTOR_ID = "CJdippxWmn9uRfooo"

PILOT_MODE = True
MAX_NEW_RUNS = 6
MAX_ITEMS = 20

ROOT = Path("data_raw_penelitian")
ROOT.mkdir(parents=True, exist_ok=True)

CHECKPOINT = ROOT / "checkpoint_baru.jsonl"

# =====================================================
# KASUS PENELITIAN
# =====================================================

CASES = {
    "LOMBOK_2018": {
        "start": "2018-07-29",
        "end": "2018-09-01",
        "pilot_date": "2018-08-06",
        "queries": [
            '(Lombok OR NTB) "bencana nasional"',
            '("gempa Lombok" OR "gempa NTB") '
            '("status bencana" OR "status nasional" OR "bencana daerah")'
        ]
    },

    "PALU_2018": {
        "start": "2018-09-28",
        "end": "2018-11-01",
        "pilot_date": "2018-09-29",
        "queries": [
            '(Palu OR Donggala) "bencana nasional"',
            '("gempa Palu" OR "tsunami Palu" OR "gempa Donggala") '
            '("status bencana" OR "status nasional" OR "bencana daerah")'
        ]
    },

    "SUMATRA_2025": {
        "start": "2025-11-26",
        "end": "2026-01-01",
        "pilot_date": "2025-11-29",
        "queries": [
            '("banjir Aceh" OR "banjir Sumatra" OR "banjir Sumatera" '
            'OR "longsor Sumatera") "bencana nasional"',
            '(Aceh OR "Sumatera Utara" OR "Sumatera Barat" '
            'OR "Sumatra Utara" OR "Sumatra Barat") '
            '("status bencana" OR "status nasional" OR "bencana daerah")'
        ]
    }
}

# =====================================================
# HELPER
# =====================================================

def run_to_dict(run):
    if isinstance(run, dict):
        return run
    if hasattr(run, "model_dump"):
        return run.model_dump(by_alias=True)
    raise TypeError(f"Format Run tidak dikenali: {type(run)}")


def to_unix(dt):
    return int(dt.timestamp())


def get_windows(config):
    if PILOT_MODE:
        dates = [pd.Timestamp(config["pilot_date"])]
    else:
        dates = list(pd.date_range(
            config["start"],
            pd.Timestamp(config["end"]) - pd.Timedelta(days=1),
            freq="D"
        ))

    for day in dates:
        start_wib = day.tz_localize("Asia/Jakarta")
        end_wib = start_wib + pd.Timedelta(days=1)

        yield start_wib, end_wib


def save_jsonl(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for item in items:
            f.write(
                json.dumps(item, ensure_ascii=False, default=str)
                + "\n"
            )


# =====================================================
# MEMBACA CHECKPOINT
# =====================================================

completed = set()

if CHECKPOINT.exists():
    with open(CHECKPOINT, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                if record.get("status") == "SAVED":
                    completed.add(record["task_id"])

# =====================================================
# MENYUSUN DAFTAR TASK
# =====================================================

task_groups = []

for case_name, config in CASES.items():
    group = []

    for start_wib, end_wib in get_windows(config):
        for query_num, base_query in enumerate(
            config["queries"], start=1
        ):

            start_unix = to_unix(start_wib)
            end_unix = to_unix(end_wib)

            search_query = (
                f"{base_query} lang:in "
                f"since_time:{start_unix} "
                f"until_time:{end_unix}"
            )

            mode = "PILOT" if PILOT_MODE else "FULL"

            task_id = (
                f"{mode}_{case_name}_"
                f"{start_wib.strftime('%Y%m%d')}_"
                f"Q{query_num}"
            )

            group.append({
                "task_id": task_id,
                "case": case_name,
                "query": search_query,
                "start_utc": start_wib.tz_convert("UTC"),
                "end_utc": end_wib.tz_convert("UTC")
            })

    task_groups.append(group)

# Bergiliran antar kasus agar semua kasus terwakili
tasks = [
    task
    for row in zip_longest(*task_groups)
    for task in row
    if task is not None
]

# =====================================================
# SCRAPING BERTAHAP
# =====================================================

new_runs = 0

for task in tasks:

    if task["task_id"] in completed:
        continue

    if new_runs >= MAX_NEW_RUNS:
        print("\nBatas batch tercapai.")
        break

    new_runs += 1

    print("\n" + "=" * 70)
    print("TASK:", task["task_id"])
    print("QUERY:", task["query"])
    print("=" * 70)

    try:
        run = client.actor(ACTOR_ID).call(
            run_input={
                "searchTerms": [task["query"]],
                "maxItems": MAX_ITEMS,
                "queryType": "Latest"
            }
        )

        run_data = run_to_dict(run)

        if run_data.get("status") != "SUCCEEDED":
            raise RuntimeError(
                f"Status Actor: {run_data.get('status')}"
            )

        dataset_id = (
            run_data.get("defaultDatasetId")
            or run_data.get("default_dataset_id")
        )

        if not dataset_id:
            raise ValueError("Dataset ID tidak ditemukan.")

        items = list(
            client.dataset(dataset_id).iterate_items()
        )

        # Simpan raw terlebih dahulu, tanpa membuang data
        raw_path = ROOT / f"{task['task_id']}_raw.jsonl"
        save_jsonl(raw_path, items)

        df = pd.DataFrame(items)

        df.to_csv(
            ROOT / f"{task['task_id']}_raw.csv",
            index=False,
            encoding="utf-8-sig"
        )

        # Validasi tanggal dan jenis tweet
        if (
            not df.empty
            and "createdAt" in df.columns
            and "type" in df.columns
        ):
            df["createdAt_utc"] = pd.to_datetime(
                df["createdAt"],
                format="%a %b %d %H:%M:%S %z %Y",
                utc=True,
                errors="coerce"
            )

            mask = (
                df["type"].eq("tweet")
                & df["createdAt_utc"].ge(task["start_utc"])
                & df["createdAt_utc"].lt(task["end_utc"])
            )

            df_valid = df.loc[mask].copy()

        else:
            df_valid = pd.DataFrame()

        valid_path = ROOT / f"{task['task_id']}_valid.csv"

        df_valid.to_csv(
            valid_path,
            index=False,
            encoding="utf-8-sig"
        )

        record = {
            "task_id": task["task_id"],
            "case": task["case"],
            "status": "SAVED",
            "dataset_id": dataset_id,
            "search_query": task["query"],
            "start_utc": str(task["start_utc"]),
            "end_utc": str(task["end_utc"]),
            "total_raw": len(df),
            "total_valid": len(df_valid),
            "raw_file": str(raw_path),
            "valid_file": str(valid_path)
        }

        with open(CHECKPOINT, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

        completed.add(task["task_id"])

        print("Dataset ID  :", dataset_id)
        print("Total raw   :", len(df))
        print("Total valid :", len(df_valid))
        print("File saved  :", raw_path)

        if len(df) >= MAX_ITEMS:
            print(
                "PERINGATAN: Hasil mungkin terpotong. "
                "Perlu evaluasi pembagian waktu lebih kecil."
            )

    except Exception as e:
        print("ERROR:", type(e).__name__, str(e))
        print("Task ini dapat dicoba ulang pada eksekusi berikutnya.")

print("\n" + "=" * 70)
print("BATCH SELESAI")
print("Run baru:", new_runs)
print("Folder:", ROOT.resolve())

10. CLEANING

In [13]:
# ==========================================
# CLEANING
# ==========================================

def clean_window_v2(df_raw):

    if df_raw.empty:
        return df_raw.copy()

    # Hanya tweet asli
    if "type" not in df_raw.columns:
        return pd.DataFrame()

    df_clean = df_raw[
        df_raw["type"] == "tweet"
    ].copy()

    # Dedup berdasarkan ID
    if "id" in df_clean.columns:

        df_clean = (
            df_clean
            .drop_duplicates(
                subset="id",
                keep="first"
            )
            .reset_index(drop=True)
        )

    # Parse timestamp
    if "createdAt" in df_clean.columns:

        df_clean["createdAt"] = pd.to_datetime(
            df_clean["createdAt"],
            format="%a %b %d %H:%M:%S %z %Y",
            errors="coerce"
        )

    return df_clean

11. SAVE RAW DATA

In [14]:
# ============================================================
# SAVE RAW WINDOW
# ============================================================

import hashlib


def make_safe_filename(
    text,
    max_length=80
):

    text = str(text)

    invalid_chars = [
        "<",
        ">",
        ":",
        '"',
        "/",
        "\\",
        "|",
        "?",
        "*"
    ]

    for char in invalid_chars:
        text = text.replace(
            char,
            "_"
        )

    text = (
        text
        .replace(" ", "_")
        .replace("__", "_")
        .strip("._ ")
    )

    if len(text) > max_length:

        digest = hashlib.md5(
            str(text).encode("utf-8")
        ).hexdigest()[:10]

        text = (
            text[:max_length - 11]
            + "_"
            + digest
        )

    return text


def save_raw_window(
    df_raw,
    period,
    keyword,
    start_wib,
    end_wib
):

    period_dir = (
        RAW_DIR / period
    )

    period_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    df_to_save = (
        df_raw.copy()
    )

    # --------------------------------------------------------
    # METADATA
    # --------------------------------------------------------

    df_to_save["period"] = (
        period
    )

    df_to_save["search_keyword"] = (
        keyword
    )

    df_to_save["window_start_wib"] = (
        start_wib
    )

    df_to_save["window_end_wib"] = (
        end_wib
    )

    # --------------------------------------------------------
    # FILENAME
    # --------------------------------------------------------

    keyword_safe = (
        make_safe_filename(
            keyword
        )
    )

    start_safe = (
        make_safe_filename(
            start_wib
        )
    )

    end_safe = (
        make_safe_filename(
            end_wib
        )
    )

    filename = (
        f"{start_safe}__"
        f"{end_safe}__"
        f"{keyword_safe}.csv"
    )

    filepath = (
        period_dir / filename
    )

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    df_to_save.to_csv(
        filepath,
        index=False,
        encoding="utf-8-sig"
    )

    return filepath

12. UPDATE CHECKPOINT

In [15]:
# ============================================================
# UPDATE CHECKPOINT
# ============================================================

def update_checkpoint(
    checkpoint,
    period,
    keyword,
    start_wib,
    end_wib,
    status,
    rows_raw=0,
    rows_unique=0,
    mock_count=0,
    dataset_id="",
    search_query="",
    since_time="",
    until_time="",
    raw_file="",
    window_minutes=0,
    error=""
):

    key = checkpoint_key(
        period,
        keyword,
        start_wib,
        end_wib
    )

    # Hitung window_minutes jika kosong
    if not window_minutes:
        start_dt = datetime.strptime(
            start_wib,
            "%Y-%m-%d %H:%M:%S"
        )

        end_dt = datetime.strptime(
            end_wib,
            "%Y-%m-%d %H:%M:%S"
        )

        window_minutes = int(
            (end_dt - start_dt).total_seconds()
            / 60
        )

    new_row = {
        "checkpoint_key": key,
        "period": period,
        "keyword": keyword,
        "window_start_wib": start_wib,
        "window_end_wib": end_wib,
        "window_minutes": int(window_minutes),
        "status": status,
        "rows_raw": int(rows_raw),
        "rows_unique": int(rows_unique),
        "mock_count": int(mock_count),
        "dataset_id": str(dataset_id),
        "search_query": str(search_query),
        "since_time": str(since_time),
        "until_time": str(until_time),
        "raw_file": str(raw_file),
        "completed_at": datetime.now().isoformat(),
        "error": str(error)
    }

    # Hapus logical window lama
    if not checkpoint.empty:

        existing_mask = (
            checkpoint["checkpoint_key"]
            .astype(str)
            .str.strip()
            == key
        )

        checkpoint = checkpoint.loc[
            ~existing_mask
        ].copy()

    # Tambahkan record terbaru
    checkpoint = pd.concat(
        [
            checkpoint,
            pd.DataFrame([new_row])
        ],
        ignore_index=True
    )

    # Safety net
    checkpoint = (
        checkpoint
        .drop_duplicates(
            subset="checkpoint_key",
            keep="last"
        )
        .reset_index(drop=True)
    )

    save_checkpoint(checkpoint)

    return checkpoint

13. METADATA PENELITIAN

In [16]:
# ==========================================
# METADATA
# ==========================================

def add_metadata(
    df,
    period,
    keyword,
    start_wib,
    end_wib
):

    df = df.copy()

    df["period"] = period
    df["search_keyword"] = keyword
    df["window_start_wib"] = start_wib
    df["window_end_wib"] = end_wib

    return df

14. ADAPTIVE SCRAPER UTAMA

In [17]:
# ==========================================
# ADAPTIVE SCRAPER UTAMA
# Resume split tanpa scrape ulang parent
# ==========================================

MAX_RETRIES = 3
RETRY_DELAY_SECONDS = 5
REQUEST_DELAY_SECONDS = 2


def log_failed_window(
    period,
    keyword,
    start_wib,
    end_wib,
    window_minutes,
    error_message
):
    """
    Menyimpan window yang gagal ke failed_windows.csv.
    """

    failed_row = {
        "period": period,
        "keyword": keyword,
        "window_start_wib": start_wib,
        "window_end_wib": end_wib,
        "window_minutes": window_minutes,
        "failed_at": datetime.now().isoformat(),
        "error": error_message
    }

    if FAILED_FILE.exists():
        failed_df = pd.read_csv(FAILED_FILE)
    else:
        failed_df = pd.DataFrame()

    failed_df = pd.concat(
        [
            failed_df,
            pd.DataFrame([failed_row])
        ],
        ignore_index=True
    )

    failed_df.to_csv(
        FAILED_FILE,
        index=False,
        encoding="utf-8-sig"
    )


def adaptive_scrape_window(
    period,
    keyword,
    start_wib,
    end_wib,
    window_minutes,
    checkpoint
):

    print("\n" + "=" * 70)
    print("ADAPTIVE SCRAPER V3")
    print("=" * 70)
    print("Period :", period)
    print("Keyword:", keyword)
    print("Window :", start_wib)
    print("      →", end_wib)
    print("Size   :", window_minutes, "menit")
    print("=" * 70)

    # ==========================================
    # CEK STATUS CHECKPOINT
    # ==========================================

    current_status = get_window_status(
        checkpoint=checkpoint,
        period=period,
        keyword=keyword,
        start_wib=start_wib,
        end_wib=end_wib
    )

    print("Status existing :", current_status)

    # ==========================================
    # 1. SUDAH SELESAI → SKIP
    # ==========================================

    if current_status in COMPLETED_STATUSES:

        print("⏭️ Window sudah completed. Skip.")

        return checkpoint

    # ==========================================
    # 2. SUDAH SPLIT → JANGAN SCRAPE ULANG
    # ==========================================

    if current_status == "split":

        print()
        print("↪️ Window sudah berstatus SPLIT.")
        print("Tidak melakukan scrape ulang parent window.")
        print("➡️ Melanjutkan langsung ke child windows.")

        smaller_window = WINDOW_LEVELS.get(
            window_minutes
        )

        if smaller_window is None:

            print(
                "⚠️ Tidak ada level window lebih kecil."
            )

            return checkpoint

        print(
            f"Child window: {smaller_window} menit"
        )

        current = datetime.strptime(
            start_wib,
            "%Y-%m-%d %H:%M:%S"
        )

        end = datetime.strptime(
            end_wib,
            "%Y-%m-%d %H:%M:%S"
        )

        delta = timedelta(
            minutes=smaller_window
        )

        while current < end:

            sub_start = current
            sub_end = min(
                current + delta,
                end
            )

            sub_start_str = sub_start.strftime(
                "%Y-%m-%d %H:%M:%S"
            )

            sub_end_str = sub_end.strftime(
                "%Y-%m-%d %H:%M:%S"
            )

            checkpoint = adaptive_scrape_window(
                period=period,
                keyword=keyword,
                start_wib=sub_start_str,
                end_wib=sub_end_str,
                window_minutes=smaller_window,
                checkpoint=checkpoint
            )

            current = sub_end

        return checkpoint

    # ==========================================
    # 3. BELUM ADA CHECKPOINT / FAILED
    # → SCRAPE
    # ==========================================

    print()

    if current_status is None:
        print("🆕 Window belum memiliki checkpoint.")
        print("➡️ Mulai scraping.")

    elif current_status == "failed":
        print("🔄 Window sebelumnya FAILED.")
        print("➡️ Mencoba scraping kembali.")

    else:
        print(
            f"ℹ️ Status: {current_status}"
        )
        print("➡️ Melanjutkan scraping.")

    # ==========================================
    # SCRAPING DENGAN RETRY
    # ==========================================

    result = None
    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        print()
        print(
            f"🔄 Percobaan {attempt}/{MAX_RETRIES}"
        )

        try:

            result = scrape_window_v3(
                search_query_base=keyword,
                start_wib=start_wib,
                end_wib=end_wib,
                max_items=MAX_ITEMS
            )

            print("✅ Scraping berhasil.")

            break

        except Exception as e:

            last_error = e

            error_message = (
                f"{type(e).__name__}: {str(e)}"
            )

            print(
                "❌ Scraping gagal:",
                error_message
            )

            if attempt < MAX_RETRIES:

                print(
                    f"⏳ Tunggu "
                    f"{RETRY_DELAY_SECONDS} detik..."
                )

                time.sleep(
                    RETRY_DELAY_SECONDS
                )

    # ==========================================
    # SEMUA RETRY GAGAL
    # ==========================================

    if result is None:

        error_message = (
            f"{type(last_error).__name__}: "
            f"{str(last_error)}"
        )

        print()
        print(
            "❌ SEMUA RETRY GAGAL"
        )
        print(
            "Error:",
            error_message
        )

        checkpoint = update_checkpoint(
            checkpoint=checkpoint,
            period=period,
            keyword=keyword,
            start_wib=start_wib,
            end_wib=end_wib,
            status="failed",
            window_minutes=window_minutes,
            error=error_message
        )

        log_failed_window(
            period=period,
            keyword=keyword,
            start_wib=start_wib,
            end_wib=end_wib,
            window_minutes=window_minutes,
            error_message=error_message
        )

        return checkpoint

    # ==========================================
    # AMBIL HASIL
    # ==========================================

    df_raw = result["df_raw"]
    dataset_id = result["dataset_id"]
    search_query = result["search_query"]
    since_time = result["since_time"]
    until_time = result["until_time"]

    # ==========================================
    # CLEAN
    # ==========================================

    df_clean = clean_window_v2(
        df_raw
    )

    rows_raw = len(df_raw)

    if (
        not df_clean.empty
        and "id" in df_clean.columns
    ):

        rows_unique = (
            df_clean["id"]
            .nunique()
        )

    else:

        rows_unique = 0

    # ==========================================
    # MOCK COUNT
    # ==========================================

    mock_count = 0

    if (
        not df_raw.empty
        and "type" in df_raw.columns
    ):

        mock_count = int(
            (
                df_raw["type"]
                == "mock_tweet"
            ).sum()
        )

    # ==========================================
    # SAVE RAW
    # ==========================================

    raw_file = save_raw_window(
        df_raw=df_raw,
        period=period,
        keyword=keyword,
        start_wib=start_wib,
        end_wib=end_wib
    )

    # ==========================================
    # OUTPUT
    # ==========================================

    print()
    print("HASIL WINDOW")
    print("-" * 40)
    print("Raw rows      :", rows_raw)
    print("Mock tweets   :", mock_count)
    print("Unique tweets :", rows_unique)
    print("Dataset ID    :", dataset_id)
    print("Raw file      :", raw_file)

    # ==========================================
    # ADAPTIVE DECISION
    # ==========================================

    if (
        rows_unique >= ADAPTIVE_THRESHOLD
        and window_minutes in WINDOW_LEVELS
    ):

        smaller_window = WINDOW_LEVELS[
            window_minutes
        ]

        print()
        print(
            f"⚠️ {rows_unique} unique tweets."
        )

        print(
            f"Window {window_minutes} menit "
            "terlalu padat."
        )

        print(
            f"➡️ Split menjadi "
            f"{smaller_window} menit."
        )

        # --------------------------------------
        # SIMPAN STATUS SPLIT
        # --------------------------------------

        checkpoint = update_checkpoint(
            checkpoint=checkpoint,
            period=period,
            keyword=keyword,
            start_wib=start_wib,
            end_wib=end_wib,
            status="split",
            rows_raw=rows_raw,
            rows_unique=rows_unique,
            mock_count=mock_count,
            dataset_id=dataset_id,
            search_query=search_query,
            since_time=since_time,
            until_time=until_time,
            raw_file=raw_file,
            window_minutes=window_minutes
        )

        # --------------------------------------
        # PROSES CHILD WINDOWS
        # --------------------------------------

        current = datetime.strptime(
            start_wib,
            "%Y-%m-%d %H:%M:%S"
        )

        end = datetime.strptime(
            end_wib,
            "%Y-%m-%d %H:%M:%S"
        )

        delta = timedelta(
            minutes=smaller_window
        )

        while current < end:

            sub_start = current

            sub_end = min(
                current + delta,
                end
            )

            sub_start_str = (
                sub_start.strftime(
                    "%Y-%m-%d %H:%M:%S"
                )
            )

            sub_end_str = (
                sub_end.strftime(
                    "%Y-%m-%d %H:%M:%S"
                )
            )

            checkpoint = adaptive_scrape_window(
                period=period,
                keyword=keyword,
                start_wib=sub_start_str,
                end_wib=sub_end_str,
                window_minutes=smaller_window,
                checkpoint=checkpoint
            )

            current = sub_end

            time.sleep(
                REQUEST_DELAY_SECONDS
            )

        return checkpoint

    # ==========================================
    # WINDOW 15 MENIT MASIH PADAT
    # ==========================================

    if (
        rows_unique >= ADAPTIVE_THRESHOLD
        and window_minutes == 15
    ):

        print()
        print(
            "⚠️ WARNING:"
        )

        print(
            "Window minimum 15 menit "
            "masih terlalu padat."
        )

        print(
            f"Unique tweets: {rows_unique}"
        )

        print(
            "➡️ Tidak di-split lagi."
        )

        print(
            "➡️ Status: completed_min_window"
        )

        checkpoint = update_checkpoint(
            checkpoint=checkpoint,
            period=period,
            keyword=keyword,
            start_wib=start_wib,
            end_wib=end_wib,
            status="completed_min_window",
            rows_raw=rows_raw,
            rows_unique=rows_unique,
            mock_count=mock_count,
            dataset_id=dataset_id,
            search_query=search_query,
            since_time=since_time,
            until_time=until_time,
            raw_file=raw_file,
            window_minutes=window_minutes
        )

        print()
        print(
            "⚠️ Window selesai pada "
            "batas minimum 15 menit."
        )

        return checkpoint

    # ==========================================
    # WINDOW AMAN
    # ==========================================

    checkpoint = update_checkpoint(
        checkpoint=checkpoint,
        period=period,
        keyword=keyword,
        start_wib=start_wib,
        end_wib=end_wib,
        status="completed",
        rows_raw=rows_raw,
        rows_unique=rows_unique,
        mock_count=mock_count,
        dataset_id=dataset_id,
        search_query=search_query,
        since_time=since_time,
        until_time=until_time,
        raw_file=raw_file,
        window_minutes=window_minutes
    )

    print()
    print(
        "✅ Window completed."
    )

    return checkpoint

In [18]:
# ============================================================
# PILOT SCRAPING TEST
# 1 CASE + 1 QUERY + 1 WINDOW
# ============================================================

PILOT_CASE = "palu_2018"
PILOT_QUERY_INDEX = 0

PILOT_START_WIB = "2018-09-28 18:00:00"
PILOT_END_WIB   = "2018-09-28 19:00:00"

config = DISASTER_CASES[PILOT_CASE]

pilot_query = config["queries"][PILOT_QUERY_INDEX]

print("=" * 80)
print("PILOT SCRAPING TEST")
print("=" * 80)

print("Case      :", PILOT_CASE)
print("Label     :", config["label"])
print("Query     :", PILOT_QUERY_INDEX + 1)
print("Window    :", PILOT_START_WIB)
print("         →", PILOT_END_WIB)
print()
print("Query:")
print(pilot_query)
print("=" * 80)

checkpoint = load_checkpoint()

checkpoint = adaptive_scrape_window(
    period=PILOT_CASE,
    keyword=pilot_query,
    start_wib=PILOT_START_WIB,
    end_wib=PILOT_END_WIB,
    window_minutes=60,
    checkpoint=checkpoint
)

print()
print("=" * 80)
print("PILOT SCRAPING SELESAI")
print("=" * 80)

print(
    checkpoint[
        [
            "period",
            "keyword",
            "window_start_wib",
            "window_end_wib",
            "status",
            "rows_raw",
            "rows_unique",
            "dataset_id",
            "raw_file"
        ]
    ].tail(1).to_string(index=False)
)

PILOT SCRAPING TEST
Case      : palu_2018
Label     : Gempa & Tsunami Palu 2018
Query     : 1
Window    : 2018-09-28 18:00:00
         → 2018-09-28 19:00:00

Query:
("Palu" OR "gempa Palu" OR "gempa bumi Palu" OR "tsunami Palu" OR "gempa tsunami Palu") AND ("bencana nasional" OR "status bencana nasional" OR "status nasional" OR "penetapan bencana nasional" OR "penetapan status bencana" OR "penetapan status")

ADAPTIVE SCRAPER V3
Period : palu_2018
Keyword: ("Palu" OR "gempa Palu" OR "gempa bumi Palu" OR "tsunami Palu" OR "gempa tsunami Palu") AND ("bencana nasional" OR "status bencana nasional" OR "status nasional" OR "penetapan bencana nasional" OR "penetapan status bencana" OR "penetapan status")
Window : 2018-09-28 18:00:00
      → 2018-09-28 19:00:00
Size   : 60 menit
Status existing : None

🆕 Window belum memiliki checkpoint.
➡️ Mulai scraping.

🔄 Percobaan 1/3

Search query:
("Palu" OR "gempa Palu" OR "gempa bumi Palu" OR "tsunami Palu" OR "gempa tsunami Palu") AND ("bencana nasi

In [19]:
# ============================================================
# CEK ERROR PILOT TERAKHIR
# ============================================================

failed_df = pd.read_csv(
    FAILED_FILE
)

print("=" * 100)
print("FAILED WINDOW TERAKHIR")
print("=" * 100)

print(
    failed_df.tail(5).to_string(
        index=False
    )
)

FAILED WINDOW TERAKHIR
   period                                                                                                                                                                                                                                                keyword    window_start_wib      window_end_wib  window_minutes                  failed_at                                             error
palu_2018 ("Palu" OR "gempa Palu" OR "gempa bumi Palu" OR "tsunami Palu" OR "gempa tsunami Palu") AND ("bencana nasional" OR "status bencana nasional" OR "status nasional" OR "penetapan bencana nasional" OR "penetapan status bencana" OR "penetapan status") 2018-09-28 18:00:00 2018-09-28 19:00:00              60 2026-10-07T22:32:49.664727 NotFoundError: Actor with this name was not found


15. RELOAD CHECKPOINT

In [19]:
# ==========================================
# RELOAD CHECKPOINT
# ==========================================

checkpoint = load_checkpoint()

print("Checkpoint sekarang:", len(checkpoint))

print(
    checkpoint[
        [
            "period",
            "keyword",
            "window_start_wib",
            "window_end_wib",
            "status",
            "rows_unique"
        ]
    ].tail(10)
)

Checkpoint sekarang: 28
     period     keyword     window_start_wib       window_end_wib     status  \
18  bencana  gempa Palu  2018-09-28 18:00:00  2018-09-28 19:00:00  completed   
19  bencana  gempa Palu  2018-09-28 19:00:00  2018-09-28 20:00:00  completed   
20  bencana  gempa Palu  2018-09-28 21:00:00  2018-09-28 22:00:00  completed   
21  bencana  gempa Palu  2018-09-28 22:00:00  2018-09-28 23:00:00  completed   
22  bencana  gempa Palu  2018-09-28 23:00:00  2018-09-29 00:00:00  completed   
23  bencana  gempa Palu  2018-09-28 20:00:00  2018-09-28 20:30:00  completed   
24  bencana  gempa Palu  2018-09-28 20:30:00  2018-09-28 20:45:00  completed   
25  bencana  gempa Palu  2018-09-28 20:45:00  2018-09-28 21:00:00  completed   
26  bencana  gempa Palu  2018-09-28 20:00:00  2018-09-28 21:00:00      split   
27  bencana  gempa Palu  2018-09-28 20:30:00  2018-09-28 21:00:00      split   

    rows_unique  
18           22  
19           62  
20          100  
21          100  
22   

16.VERIVIKASI CHECKPOINT

In [20]:
# ==========================================
# VERIFIKASI HASIL ADAPTIVE
# ==========================================

checkpoint = load_checkpoint()

target_windows = [
    (
        "2018-09-28 20:00:00",
        "2018-09-28 21:00:00",
        60,
        "split"
    ),
    (
        "2018-09-28 20:00:00",
        "2018-09-28 20:30:00",
        30,
        "completed"
    ),
    (
        "2018-09-28 20:30:00",
        "2018-09-28 21:00:00",
        30,
        "split"
    ),
    (
        "2018-09-28 20:30:00",
        "2018-09-28 20:45:00",
        15,
        "completed"
    ),
    (
        "2018-09-28 20:45:00",
        "2018-09-28 21:00:00",
        15,
        "completed"
    )
]

print("=" * 90)
print("VERIFIKASI ADAPTIVE SCRAPER V2")
print("=" * 90)

all_valid = True

for (
    start_time,
    end_time,
    expected_minutes,
    expected_status
) in target_windows:

    mask = (
        (checkpoint["period"] == "bencana") &
        (checkpoint["keyword"] == "gempa Palu") &
        (
            checkpoint["window_start_wib"].astype(str)
            == start_time
        ) &
        (
            checkpoint["window_end_wib"].astype(str)
            == end_time
        )
    )

    result = checkpoint.loc[mask]

    if result.empty:

        print()
        print(
            f"❌ {start_time} → {end_time}"
        )
        print(
            "   Checkpoint tidak ditemukan."
        )

        all_valid = False
        continue

    row = result.iloc[-1]

    actual_minutes = int(
        row["window_minutes"]
    )

    actual_status = str(
        row["status"]
    ).strip()

    actual_unique = int(
        row["rows_unique"]
    )

    print()
    print(
        f"Window : {start_time} → {end_time}"
    )

    print(
        f"Size   : {actual_minutes} menit"
    )

    print(
        f"Status : {actual_status}"
    )

    print(
        f"Unique : {actual_unique}"
    )

    if (
        actual_minutes == expected_minutes
        and actual_status == expected_status
    ):

        print("   ✅ VALID")

    else:

        print(
            f"   ❌ INVALID "
            f"(expected {expected_minutes} menit / "
            f"{expected_status})"
        )

        all_valid = False


print()
print("=" * 90)
print("HASIL AKHIR")
print("=" * 90)

if all_valid:

    print(
        "✅ SEMUA WINDOW ADAPTIVE VALID."
    )

    print(
        "✅ Parent split tersimpan dengan benar."
    )

    print(
        "✅ Child window tersimpan dengan benar."
    )

    print(
        "✅ Resume logic dapat membedakan "
        "parent dan child."
    )

else:

    print(
        "⚠️ MASIH ADA WINDOW YANG PERLU DIPERIKSA."
    )

print("=" * 90)

VERIFIKASI ADAPTIVE SCRAPER V2

Window : 2018-09-28 20:00:00 → 2018-09-28 21:00:00
Size   : 60 menit
Status : split
Unique : 100
   ✅ VALID

Window : 2018-09-28 20:00:00 → 2018-09-28 20:30:00
Size   : 30 menit
Status : completed
Unique : 57
   ✅ VALID

Window : 2018-09-28 20:30:00 → 2018-09-28 21:00:00
Size   : 30 menit
Status : split
Unique : 95
   ✅ VALID

Window : 2018-09-28 20:30:00 → 2018-09-28 20:45:00
Size   : 15 menit
Status : completed
Unique : 50
   ✅ VALID

Window : 2018-09-28 20:45:00 → 2018-09-28 21:00:00
Size   : 15 menit
Status : completed
Unique : 45
   ✅ VALID

HASIL AKHIR
✅ SEMUA WINDOW ADAPTIVE VALID.
✅ Parent split tersimpan dengan benar.
✅ Child window tersimpan dengan benar.
✅ Resume logic dapat membedakan parent dan child.


17. CLEAN LEGACY

In [21]:
# ============================================================
# CLEAN LEGACY CHECKPOINT
# ============================================================

checkpoint = load_checkpoint()

print("Checkpoint sebelum cleaning :", len(checkpoint))

# Pastikan checkpoint_key dibangun ulang
checkpoint["checkpoint_key"] = (
    checkpoint["period"].astype(str)
    + "|"
    + checkpoint["keyword"].astype(str)
    + "|"
    + checkpoint["window_start_wib"].astype(str)
    + "|"
    + checkpoint["window_end_wib"].astype(str)
)

# Hapus baris yang tidak memiliki informasi window lengkap
valid_mask = (
    (checkpoint["period"].astype(str).str.strip() != "") &
    (checkpoint["keyword"].astype(str).str.strip() != "") &
    (checkpoint["window_start_wib"].astype(str).str.strip() != "") &
    (checkpoint["window_end_wib"].astype(str).str.strip() != "")
)

checkpoint = checkpoint.loc[valid_mask].copy()

# Untuk checkpoint dengan window yang sama,
# pertahankan record terakhir.
checkpoint = (
    checkpoint
    .drop_duplicates(
        subset="checkpoint_key",
        keep="last"
    )
    .reset_index(drop=True)
)

save_checkpoint(checkpoint)

print("Checkpoint setelah cleaning :", len(checkpoint))
print()
print(
    checkpoint[
        [
            "period",
            "keyword",
            "window_start_wib",
            "window_end_wib",
            "window_minutes",
            "status",
            "rows_unique"
        ]
    ].tail(20).to_string(index=False)
)

Checkpoint sebelum cleaning : 28
Checkpoint setelah cleaning : 28

 period    keyword    window_start_wib      window_end_wib  window_minutes    status  rows_unique
bencana gempa Palu 2018-09-28 08:00:00 2018-09-28 09:00:00              60 completed            0
bencana gempa Palu 2018-09-28 09:00:00 2018-09-28 10:00:00              60 completed            0
bencana gempa Palu 2018-09-28 10:00:00 2018-09-28 11:00:00              60 completed            0
bencana gempa Palu 2018-09-28 11:00:00 2018-09-28 12:00:00              60 completed            0
bencana gempa Palu 2018-09-28 12:00:00 2018-09-28 13:00:00              60 completed            0
bencana gempa Palu 2018-09-28 13:00:00 2018-09-28 14:00:00              60 completed            0
bencana gempa Palu 2018-09-28 14:00:00 2018-09-28 15:00:00              60 completed            5
bencana gempa Palu 2018-09-28 15:00:00 2018-09-28 16:00:00              60 completed            4
bencana gempa Palu 2018-09-28 16:00:00 2018-09-28 1

In [22]:
# ============================================================
# NORMALIZE CHECKPOINT WINDOW MINUTES
# ============================================================

checkpoint = load_checkpoint()

def calculate_window_minutes(start_wib, end_wib):
    """
    Menghitung durasi window berdasarkan start dan end WIB.
    """

    try:
        start = datetime.strptime(
            str(start_wib),
            "%Y-%m-%d %H:%M:%S"
        )

        end = datetime.strptime(
            str(end_wib),
            "%Y-%m-%d %H:%M:%S"
        )

        return int(
            (end - start).total_seconds() / 60
        )

    except Exception:
        return 0


# Isi window_minutes yang kosong / NaN
checkpoint["window_minutes"] = checkpoint.apply(
    lambda row: (
        calculate_window_minutes(
            row["window_start_wib"],
            row["window_end_wib"]
        )
        if pd.isna(row["window_minutes"])
        or str(row["window_minutes"]).strip() in ["", "nan"]
        else int(float(row["window_minutes"]))
    ),
    axis=1
)


# Simpan kembali
save_checkpoint(checkpoint)

print("Checkpoint berhasil dinormalisasi.")
print()

print(
    checkpoint[
        [
            "period",
            "keyword",
            "window_start_wib",
            "window_end_wib",
            "window_minutes",
            "status",
            "rows_unique"
        ]
    ].tail(25).to_string(index=False)
)

Checkpoint berhasil dinormalisasi.

 period    keyword    window_start_wib      window_end_wib  window_minutes    status  rows_unique
bencana gempa Palu 2018-09-28 03:00:00 2018-09-28 04:00:00              60 completed            0
bencana gempa Palu 2018-09-28 04:00:00 2018-09-28 05:00:00              60 completed            0
bencana gempa Palu 2018-09-28 05:00:00 2018-09-28 06:00:00              60 completed            0
bencana gempa Palu 2018-09-28 06:00:00 2018-09-28 07:00:00              60 completed            0
bencana gempa Palu 2018-09-28 07:00:00 2018-09-28 08:00:00              60 completed            0
bencana gempa Palu 2018-09-28 08:00:00 2018-09-28 09:00:00              60 completed            0
bencana gempa Palu 2018-09-28 09:00:00 2018-09-28 10:00:00              60 completed            0
bencana gempa Palu 2018-09-28 10:00:00 2018-09-28 11:00:00              60 completed            0
bencana gempa Palu 2018-09-28 11:00:00 2018-09-28 12:00:00              60 complet

In [23]:
# ============================================================
# VALIDATE CHECKPOINT
# ============================================================

checkpoint = load_checkpoint()

print("========================================")
print("CHECKPOINT VALIDATION")
print("========================================")

print("Total checkpoint :", len(checkpoint))

# Cek checkpoint_key kosong
empty_keys = checkpoint[
    checkpoint["checkpoint_key"]
    .astype(str)
    .str.strip()
    == ""
]

print(
    "Checkpoint key kosong :",
    len(empty_keys)
)

# Cek duplicate logical window
duplicate_keys = (
    checkpoint["checkpoint_key"]
    .value_counts()
)

duplicate_keys = duplicate_keys[
    duplicate_keys > 1
]

print(
    "Duplicate checkpoint key :",
    len(duplicate_keys)
)

# Cek window_minutes kosong
empty_window_minutes = checkpoint[
    checkpoint["window_minutes"].isna()
]

print(
    "Window minutes kosong :",
    len(empty_window_minutes)
)

# Cek status
print()
print("Distribusi status:")
print(
    checkpoint["status"]
    .value_counts()
)

print()
print("Checkpoint validation selesai.")

CHECKPOINT VALIDATION
Total checkpoint : 28
Checkpoint key kosong : 0
Duplicate checkpoint key : 0
Window minutes kosong : 0

Distribusi status:
status
completed    26
split         2
Name: count, dtype: int64

Checkpoint validation selesai.


18. LOAD SEMUA DATA RAW

In [24]:
# ============================================================
# LOAD SEMUA RAW DATA
# MULTI-KEYWORD + FAIL-SAFE METADATA
# ============================================================

def parse_datetime_from_filename(text):

    formats = [
        "%Y-%m-%d_%H-%M-%S",
        "%Y-%m-%d %H-%M-%S",
        "%Y-%m-%d_%H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
    ]

    for fmt in formats:
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            continue

    return None


def parse_metadata_from_raw_file(file):

    period = file.parent.name

    parts = file.stem.split("__")

    start_wib = ""
    end_wib = ""
    keyword = ""

    # ========================================================
    # FORMAT BARU
    # START__END__KEYWORD
    # ========================================================

    if len(parts) >= 3:

        start_dt = parse_datetime_from_filename(parts[0])
        end_dt = parse_datetime_from_filename(parts[1])

        keyword_raw = "__".join(parts[2:])

        if start_dt is not None:
            start_wib = start_dt.strftime(
                "%Y-%m-%d %H:%M:%S"
            )

        if end_dt is not None:
            end_wib = end_dt.strftime(
                "%Y-%m-%d %H:%M:%S"
            )

        keyword = (
            keyword_raw
            .replace("_", " ")
            .strip()
        )

    # ========================================================
    # FORMAT LAMA
    # START__KEYWORD
    # ========================================================

    elif len(parts) == 2:

        start_dt = parse_datetime_from_filename(parts[0])

        keyword = (
            parts[1]
            .replace("_", " ")
            .strip()
        )

        if start_dt is not None:

            start_wib = start_dt.strftime(
                "%Y-%m-%d %H:%M:%S"
            )

            # ------------------------------------------------
            # Recover END dari checkpoint
            # ------------------------------------------------

            checkpoint_local = load_checkpoint()

            mask = (
                (checkpoint_local["period"] == period)
                &
                (checkpoint_local["keyword"] == keyword)
                &
                (
                    checkpoint_local[
                        "window_start_wib"
                    ]
                    == start_wib
                )
            )

            matched = checkpoint_local.loc[mask]

            if not matched.empty:

                row = matched.iloc[-1]

                try:

                    duration = int(
                        row["window_minutes"]
                    )

                    end_dt = (
                        start_dt
                        + timedelta(
                            minutes=duration
                        )
                    )

                    end_wib = end_dt.strftime(
                        "%Y-%m-%d %H:%M:%S"
                    )

                except Exception:
                    pass

    return {
        "period": period,
        "keyword": keyword,
        "window_start_wib": start_wib,
        "window_end_wib": end_wib
    }


def fill_metadata_if_missing(
    df,
    metadata
):

    df = df.copy()

    for column, value in metadata.items():

        if column not in df.columns:

            df[column] = value

        else:

            empty_mask = (
                df[column].isna()
                |
                (
                    df[column]
                    .astype(str)
                    .str.strip()
                    == ""
                )
                |
                (
                    df[column]
                    .astype(str)
                    .str.lower()
                    == "nan"
                )
                |
                (
                    df[column]
                    .astype(str)
                    .str.lower()
                    == "none"
                )
            )

            df.loc[
                empty_mask,
                column
            ] = value

    return df


def load_all_raw_tweets_v6():

    raw_files = sorted(
        RAW_DIR.rglob("*.csv")
    )

    print("=" * 70)
    print("LOAD RAW DATA V6")
    print("=" * 70)

    print(
        "Jumlah raw file ditemukan :",
        len(raw_files)
    )

    all_data = []

    metadata_failed = []
    read_failed = []
    id_failed = []
    empty_files = []

    processed_files = []

    # ========================================================
    # LOAD FILE SATU PER SATU
    # ========================================================

    for file in raw_files:

        # ----------------------------------------------------
        # READ FILE
        # ----------------------------------------------------

        try:

            df = pd.read_csv(file)

        except Exception as e:

            read_failed.append({
                "file": str(file),
                "error": f"{type(e).__name__}: {e}"
            })

            continue

        if df.empty:

            empty_files.append(str(file))

            continue

        # ----------------------------------------------------
        # HANYA TWEET ASLI
        # ----------------------------------------------------

        if "type" in df.columns:

            df = df[
                df["type"]
                .astype(str)
                .str.lower()
                == "tweet"
            ].copy()

        if df.empty:

            empty_files.append(str(file))

            continue

        # ----------------------------------------------------
        # ID WAJIB ADA
        # ----------------------------------------------------

        if "id" not in df.columns:

            id_failed.append({
                "file": str(file),
                "reason": "Kolom id tidak ditemukan"
            })

            continue

        # Buang ID kosong
        id_empty_mask = (
            df["id"].isna()
            |
            (
                df["id"]
                .astype(str)
                .str.strip()
                == ""
            )
            |
            (
                df["id"]
                .astype(str)
                .str.lower()
                == "nan"
            )
        )

        if id_empty_mask.any():

            id_failed.append({
                "file": str(file),
                "reason": (
                    f"{int(id_empty_mask.sum())} "
                    "row memiliki ID kosong"
                )
            })

            df = df.loc[
                ~id_empty_mask
            ].copy()

        if df.empty:
            continue

        # ----------------------------------------------------
        # PARSE METADATA
        # ----------------------------------------------------

        metadata = parse_metadata_from_raw_file(
            file
        )

        valid_metadata = all(
            str(metadata[column]).strip() != ""
            for column in [
                "period",
                "keyword",
                "window_start_wib",
                "window_end_wib"
            ]
        )

        if not valid_metadata:

            metadata_failed.append({
                "file": str(file),
                "metadata": metadata
            })

            # PENTING:
            # Jangan masukkan file ke dataset penelitian.
            # Lebih baik STOP daripada silent data loss.
            continue

        # ----------------------------------------------------
        # TAMBAHKAN METADATA
        # ----------------------------------------------------

        df = fill_metadata_if_missing(
            df,
            metadata
        )

        # Keyword asli yang digunakan saat scraping
        df["scrape_keyword"] = (
            metadata["keyword"]
        )

        # File sumber
        df["source_raw_file"] = str(file)

        # ----------------------------------------------------
        # APPEND
        # ----------------------------------------------------

        all_data.append(df)

        processed_files.append(
            str(file)
        )

    # ========================================================
    # REPORT ERROR
    # ========================================================

    print()
    print("=" * 70)
    print("HASIL PEMERIKSAAN FILE")
    print("=" * 70)

    print(
        "File berhasil diproses :",
        len(processed_files)
    )

    print(
        "File gagal dibaca      :",
        len(read_failed)
    )

    print(
        "Metadata gagal         :",
        len(metadata_failed)
    )

    print(
        "File tanpa / invalid ID:",
        len(id_failed)
    )

    print(
        "File kosong            :",
        len(empty_files)
    )

    # ========================================================
    # FAIL-SAFE
    # ========================================================

    if read_failed:

        print()
        print("FILE GAGAL DIBACA:")

        for item in read_failed:
            print(
                "-",
                item["file"],
                "|",
                item["error"]
            )

        raise RuntimeError(
            "Ada raw file yang gagal dibaca."
        )

    if metadata_failed:

        print()
        print("FILE GAGAL METADATA:")

        for item in metadata_failed:
            print(
                "-",
                item["file"],
                "|",
                item["metadata"]
            )

        raise RuntimeError(
            "Ada raw file dengan metadata "
            "yang gagal diparse."
        )

    if id_failed:

        print()
        print("FILE / ROW DENGAN MASALAH ID:")

        for item in id_failed:
            print(
                "-",
                item["file"],
                "|",
                item["reason"]
            )

        raise RuntimeError(
            "Ditemukan masalah ID pada raw data."
        )

    if not all_data:

        raise RuntimeError(
            "Tidak ada raw tweet yang berhasil dimuat."
        )

    # ========================================================
    # CONCAT
    # ========================================================

    df_all_raw = pd.concat(
        all_data,
        ignore_index=True
    )

    # ========================================================
    # VALIDASI DASAR
    # ========================================================

    raw_rows = len(df_all_raw)
    unique_ids = df_all_raw["id"].nunique()

    print()
    print("=" * 70)
    print("HASIL LOAD RAW")
    print("=" * 70)

    print(
        "Jumlah file berhasil :",
        len(processed_files)
    )

    print(
        "Rows raw             :",
        raw_rows
    )

    print(
        "ID unik              :",
        unique_ids
    )

    # ========================================================
    # VALIDASI METADATA
    # ========================================================

    metadata_columns = [
        "period",
        "scrape_keyword",
        "window_start_wib",
        "window_end_wib",
        "source_raw_file"
    ]

    print()
    print("=" * 70)
    print("VALIDASI METADATA RAW")
    print("=" * 70)

    for column in metadata_columns:

        missing = (
            df_all_raw[column]
            .isna()
            .sum()
        )

        empty = (
            df_all_raw[column]
            .astype(str)
            .str.strip()
            .isin(["", "nan", "None"])
            .sum()
        )

        print(
            f"{column:22s}:",
            "missing =",
            missing,
            "| empty =",
            empty
        )

        if missing > 0 or empty > 0:

            raise RuntimeError(
                f"Metadata {column} tidak lengkap."
            )

    # ========================================================
    # DISTRIBUSI KEYWORD
    # ========================================================

    print()
    print("=" * 70)
    print("DISTRIBUSI SCRAPE KEYWORD")
    print("=" * 70)

    print(
        df_all_raw[
            "scrape_keyword"
        ].value_counts()
    )

    # ========================================================
    # DISTRIBUSI KEYWORD PER TWEET
    # ========================================================

    keyword_per_id = (
        df_all_raw
        .groupby("id")[
            "scrape_keyword"
        ]
        .nunique()
    )

    print()
    print("=" * 70)
    print("DISTRIBUSI JUMLAH KEYWORD PER TWEET")
    print("=" * 70)

    print(
        keyword_per_id
        .value_counts()
        .sort_index()
    )

    print()
    print(
        "Tweet ditemukan >1 keyword:",
        int((keyword_per_id > 1).sum())
    )

    return df_all_raw


# ============================================================
# EXECUTE
# ============================================================

df_all_raw = load_all_raw_tweets_v6()

LOAD RAW DATA V6
Jumlah raw file ditemukan : 29

HASIL PEMERIKSAAN FILE
File berhasil diproses : 15
File gagal dibaca      : 0
Metadata gagal         : 0
File tanpa / invalid ID: 0
File kosong            : 14

HASIL LOAD RAW
Jumlah file berhasil : 15
Rows raw             : 1716
ID unik              : 564

VALIDASI METADATA RAW
period                : missing = 0 | empty = 0
scrape_keyword        : missing = 0 | empty = 0
window_start_wib      : missing = 0 | empty = 0
window_end_wib        : missing = 0 | empty = 0
source_raw_file       : missing = 0 | empty = 0

DISTRIBUSI SCRAPE KEYWORD
scrape_keyword
gempa Palu    1716
Name: count, dtype: int64

DISTRIBUSI JUMLAH KEYWORD PER TWEET
scrape_keyword
1    564
Name: count, dtype: int64

Tweet ditemukan >1 keyword: 0


19. NORMALISASI TIMESTAMP

In [25]:
# ============================================================
# NORMALISASI TIMESTAMP
# ============================================================

if df_all_raw.empty:
    raise RuntimeError(
        "df_all_raw kosong."
    )

if "createdAt" not in df_all_raw.columns:
    raise RuntimeError(
        "Kolom createdAt tidak ditemukan."
    )

df_all_raw["createdAt"] = pd.to_datetime(
    df_all_raw["createdAt"],
    format="%a %b %d %H:%M:%S %z %Y",
    errors="coerce",
    utc=True
)

timestamp_nan = (
    df_all_raw["createdAt"]
    .isna()
    .sum()
)

print("=" * 60)
print("TIMESTAMP VALIDATION")
print("=" * 60)

print(
    "Timestamp NaN :",
    timestamp_nan
)

print(
    "Tanggal awal  :",
    df_all_raw["createdAt"].min()
)

print(
    "Tanggal akhir :",
    df_all_raw["createdAt"].max()
)

if timestamp_nan > 0:

    print()
    print(
        "⚠️ PERINGATAN:",
        timestamp_nan,
        "timestamp tidak dapat diparse."
    )

else:

    print(
        "✅ Semua timestamp valid."
    )

TIMESTAMP VALIDATION
Timestamp NaN : 0
Tanggal awal  : 2018-09-28 07:07:55+00:00
Tanggal akhir : 2018-09-28 16:59:04+00:00
✅ Semua timestamp valid.


20. MERGE TWEETS

In [26]:
# ============================================================
# MERGE TWEETS ANTAR-KEYWORD
# ============================================================

def combine_unique_values(series):

    values = (
        series
        .dropna()
        .astype(str)
        .str.strip()
    )

    values = [
        value
        for value in values
        if value
        and value.lower() not in {
            "nan",
            "none",
            ""
        }
    ]

    return sorted(
        set(values)
    )


def merge_tweets_across_keywords(
    df
):

    if df.empty:

        raise RuntimeError(
            "Data raw kosong."
        )

    required_columns = [
        "id",
        "scrape_keyword",
        "source_raw_file"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:

        raise ValueError(
            "Kolom wajib tidak ditemukan: "
            + ", ".join(missing_columns)
        )

    print("=" * 70)
    print("MERGE TWEETS ANTAR-KEYWORD V2")
    print("=" * 70)

    raw_rows = len(df)

    raw_unique_ids = (
        df["id"].nunique()
    )

    print(
        "Raw rows sebelum merge :",
        raw_rows
    )

    print(
        "ID unik sebelum merge  :",
        raw_unique_ids
    )

    # ========================================================
    # 1. KEYWORD MAP
    # ========================================================

    keyword_map = (
        df.groupby("id")[
            "scrape_keyword"
        ]
        .apply(
            lambda s: "; ".join(
                combine_unique_values(s)
            )
        )
        .rename(
            "matched_keywords"
        )
    )

    # ========================================================
    # 2. KEYWORD COUNT
    # ========================================================

    keyword_count_map = (
        df.groupby("id")[
            "scrape_keyword"
        ]
        .nunique()
        .rename(
            "keyword_count"
        )
    )

    # ========================================================
    # 3. SOURCE RAW FILE MAP
    # ========================================================

    source_map = (
        df.groupby("id")[
            "source_raw_file"
        ]
        .apply(
            lambda s: "; ".join(
                combine_unique_values(s)
            )
        )
        .rename(
            "matched_raw_files"
        )
    )

    # ========================================================
    # 4. WINDOW MAP
    # ========================================================

    window_start_map = (
        df.groupby("id")[
            "window_start_wib"
        ]
        .apply(
            lambda s: "; ".join(
                combine_unique_values(s)
            )
        )
        .rename(
            "matched_window_start_wib"
        )
    )

    window_end_map = (
        df.groupby("id")[
            "window_end_wib"
        ]
        .apply(
            lambda s: "; ".join(
                combine_unique_values(s)
            )
        )
        .rename(
            "matched_window_end_wib"
        )
    )

    # ========================================================
    # 5. BASE TWEET
    # ========================================================

    # Satu tweet = satu row.
    # Data tweet diambil dari kemunculan pertama.
    df_unique = (
        df
        .drop_duplicates(
            subset="id",
            keep="first"
        )
        .copy()
    )

    # ========================================================
    # 6. HAPUS KOLOM METADATA AGREGAT LAMA
    # ========================================================

    columns_to_drop = [
        "matched_keywords",
        "keyword_count",
        "matched_raw_files",
        "matched_window_start_wib",
        "matched_window_end_wib"
    ]

    df_unique = df_unique.drop(
        columns=[
            column
            for column in columns_to_drop
            if column in df_unique.columns
        ],
        errors="ignore"
    )

    # ========================================================
    # 7. JOIN HASIL AGREGASI
    # ========================================================

    df_unique = (
        df_unique
        .join(keyword_map, on="id")
        .join(keyword_count_map, on="id")
        .join(source_map, on="id")
        .join(window_start_map, on="id")
        .join(window_end_map, on="id")
    )

    # ========================================================
    # 8. NORMALISASI URUTAN KOLOM
    # ========================================================

    preferred_columns = [
        "id",
        "createdAt",
        "fullText",
        "lang",
        "username",
        "name",
        "likeCount",
        "retweetCount",
        "replyCount",
        "quoteCount",
        "viewCount",
        "period",
        "scrape_keyword",
        "matched_keywords",
        "keyword_count",
        "source_raw_file",
        "matched_raw_files",
        "window_start_wib",
        "window_end_wib",
        "matched_window_start_wib",
        "matched_window_end_wib"
    ]

    existing_preferred = [
        column
        for column in preferred_columns
        if column in df_unique.columns
    ]

    remaining_columns = [
        column
        for column in df_unique.columns
        if column not in existing_preferred
    ]

    df_unique = df_unique[
        existing_preferred
        + remaining_columns
    ].copy()

    # ========================================================
    # 9. FINAL MERGE VALIDATION
    # ========================================================

    final_rows = len(df_unique)

    final_unique_ids = (
        df_unique["id"].nunique()
    )

    print()
    print("=" * 70)
    print("MERGE BERHASIL")
    print("=" * 70)

    print(
        "Raw rows  :",
        raw_rows
    )

    print(
        "Final rows:",
        final_rows
    )

    print(
        "ID unik   :",
        final_unique_ids
    )

    # --------------------------------------------------------
    # Konsistensi jumlah row
    # --------------------------------------------------------

    if final_rows != raw_unique_ids:

        raise RuntimeError(
            "Jumlah final rows tidak sama "
            "dengan jumlah ID unik sebelum merge."
        )

    if final_unique_ids != final_rows:

        raise RuntimeError(
            "Masih terdapat duplicate ID "
            "setelah merge."
        )

    # --------------------------------------------------------
    # Keyword map tidak boleh kosong
    # --------------------------------------------------------

    empty_keyword_map = (
        df_unique["matched_keywords"]
        .isna()
        |
        (
            df_unique["matched_keywords"]
            .astype(str)
            .str.strip()
            == ""
        )
    )

    if empty_keyword_map.any():

        raise RuntimeError(
            "Ada tweet tanpa matched_keywords."
        )

    # --------------------------------------------------------
    # Keyword count consistency
    # --------------------------------------------------------

    calculated_keyword_count = (
        df_unique["matched_keywords"]
        .astype(str)
        .apply(
            lambda x: len(
                [
                    item
                    for item in x.split(";")
                    if item.strip()
                ]
            )
        )
    )

    mismatch = (
        calculated_keyword_count
        != df_unique["keyword_count"]
    )

    if mismatch.any():

        print()
        print(
            "Jumlah mismatch keyword_count:",
            int(mismatch.sum())
        )

        raise RuntimeError(
            "keyword_count tidak konsisten "
            "dengan matched_keywords."
        )

    print()
    print(
        "Tweet dengan >1 keyword:",
        int(
            (
                df_unique["keyword_count"]
                > 1
            ).sum()
        )
    )

    return df_unique


# ============================================================
# EXECUTE
# ============================================================

df_merged = merge_tweets_across_keywords(
    df_all_raw
)

MERGE TWEETS ANTAR-KEYWORD V2
Raw rows sebelum merge : 1716
ID unik sebelum merge  : 564

MERGE BERHASIL
Raw rows  : 1716
Final rows: 564
ID unik   : 564

Tweet dengan >1 keyword: 0


21. VALIDASI DATASET FINAL

In [27]:
# ============================================================
# VALIDASI DATASET FINAL
# ============================================================

def validate_final_dataset(
    df_final,
    df_raw=None
):

    print("=" * 70)
    print("VALIDASI DATASET FINAL V2")
    print("=" * 70)

    if df_final is None:
        raise RuntimeError(
            "df_final belum tersedia."
        )

    if df_final.empty:
        raise RuntimeError(
            "Dataset final kosong."
        )

    # ========================================================
    # 1. ROW & ID
    # ========================================================

    total_rows = len(df_final)

    unique_ids = (
        df_final["id"].nunique()
    )

    duplicate_ids = (
        df_final["id"]
        .duplicated()
        .sum()
    )

    print(
        "Jumlah rows final    :",
        total_rows
    )

    print(
        "Jumlah ID unik       :",
        unique_ids
    )

    print(
        "ID masih duplicate   :",
        duplicate_ids
    )

    if duplicate_ids > 0:

        raise RuntimeError(
            "Masih terdapat duplicate ID."
        )

    # ========================================================
    # 2. ID KOSONG
    # ========================================================

    empty_id = (
        df_final["id"].isna()
        |
        (
            df_final["id"]
            .astype(str)
            .str.strip()
            == ""
        )
        |
        (
            df_final["id"]
            .astype(str)
            .str.lower()
            == "nan"
        )
    )

    print(
        "ID kosong            :",
        int(empty_id.sum())
    )

    if empty_id.any():

        raise RuntimeError(
            "Ditemukan ID kosong."
        )

    # ========================================================
    # 3. MOCK TWEET
    # ========================================================

    mock_tweets = 0

    if "type" in df_final.columns:

        mock_tweets = int(
            (
                df_final["type"]
                .astype(str)
                .str.lower()
                == "mock_tweet"
            ).sum()
        )

    print(
        "Mock tweet            :",
        mock_tweets
    )

    if mock_tweets > 0:

        raise RuntimeError(
            "Dataset final masih mengandung mock tweet."
        )

    # ========================================================
    # 4. TIMESTAMP
    # ========================================================

    if "createdAt" not in df_final.columns:

        raise RuntimeError(
            "Kolom createdAt tidak ditemukan."
        )

    timestamp_nan = int(
        df_final["createdAt"]
        .isna()
        .sum()
    )

    print(
        "Timestamp NaN         :",
        timestamp_nan
    )

    if timestamp_nan > 0:

        raise RuntimeError(
            "Ada timestamp NaN."
        )

    # ========================================================
    # 5. KEYWORD METADATA
    # ========================================================

    required_metadata = [
        "matched_keywords",
        "keyword_count",
        "source_raw_file",
        "matched_raw_files",
        "matched_window_start_wib",
        "matched_window_end_wib"
    ]

    for column in required_metadata:

        if column not in df_final.columns:

            raise RuntimeError(
                f"Kolom wajib tidak ditemukan: {column}"
            )

    # ========================================================
    # 6. METADATA KOSONG
    # ========================================================

    print()
    print("VALIDASI METADATA")
    print("-" * 70)

    for column in required_metadata:

        empty_count = int(
            df_final[column]
            .isna()
            .sum()
        )

        string_empty_count = int(
            (
                df_final[column]
                .astype(str)
                .str.strip()
                .isin(
                    ["", "nan", "None"]
                )
            ).sum()
        )

        total_empty = max(
            empty_count,
            string_empty_count
        )

        print(
            f"{column:30s}:",
            total_empty
        )

        if total_empty > 0:

            raise RuntimeError(
                f"Metadata {column} kosong."
            )

    # ========================================================
    # 7. KEYWORD COUNT CONSISTENCY
    # ========================================================

    calculated_keyword_count = (
        df_final["matched_keywords"]
        .astype(str)
        .apply(
            lambda x: len(
                [
                    item
                    for item in x.split(";")
                    if item.strip()
                ]
            )
        )
    )

    keyword_count_mismatch = (
        calculated_keyword_count
        != df_final["keyword_count"]
    )

    mismatch_count = int(
        keyword_count_mismatch.sum()
    )

    print()
    print(
        "keyword_count mismatch:",
        mismatch_count
    )

    if mismatch_count > 0:

        raise RuntimeError(
            "keyword_count tidak konsisten."
        )

    # ========================================================
    # 8. DISTRIBUSI KEYWORD COUNT
    # ========================================================

    print()
    print("Distribusi keyword_count:")
    print(
        df_final[
            "keyword_count"
        ]
        .value_counts()
        .sort_index()
    )

    # ========================================================
    # 9. CROSS-CHECK RAW
    # ========================================================

    if df_raw is not None:

        raw_unique_ids = (
            df_raw["id"].nunique()
        )

        final_unique_ids = (
            df_final["id"].nunique()
        )

        print()
        print("CROSS-CHECK RAW")
        print("-" * 70)

        print(
            "Raw unique ID  :",
            raw_unique_ids
        )

        print(
            "Final unique ID:",
            final_unique_ids
        )

        if (
            raw_unique_ids
            != final_unique_ids
        ):

            raise RuntimeError(
                "Jumlah ID unik raw dan final berbeda."
            )

    # ========================================================
    # 10. HASIL AKHIR
    # ========================================================

    print()
    print("=" * 70)
    print("VALIDASI SELESAI")
    print("=" * 70)

    print(
        "✅ Dataset final tidak memiliki duplicate ID."
    )

    print(
        "✅ Tidak ada ID kosong."
    )

    print(
        "✅ Tidak ada mock tweet."
    )

    print(
        "✅ Semua timestamp valid."
    )

    print(
        "✅ Metadata keyword lengkap."
    )

    print(
        "✅ keyword_count konsisten."
    )

    print(
        "✅ Jumlah ID raw dan final konsisten."
    )

    print(
        "✅ Dataset siap untuk tahap penyimpanan."
    )

    return True


# ============================================================
# EXECUTE
# ============================================================

validate_final_dataset(
    df_final=df_merged,
    df_raw=df_all_raw
)

VALIDASI DATASET FINAL V2
Jumlah rows final    : 564
Jumlah ID unik       : 564
ID masih duplicate   : 0
ID kosong            : 0
Mock tweet            : 0
Timestamp NaN         : 0

VALIDASI METADATA
----------------------------------------------------------------------
matched_keywords              : 0
keyword_count                 : 0
source_raw_file               : 0
matched_raw_files             : 0
matched_window_start_wib      : 0
matched_window_end_wib        : 0

keyword_count mismatch: 0

Distribusi keyword_count:
keyword_count
1    564
Name: count, dtype: int64

CROSS-CHECK RAW
----------------------------------------------------------------------
Raw unique ID  : 564
Final unique ID: 564

VALIDASI SELESAI
✅ Dataset final tidak memiliki duplicate ID.
✅ Tidak ada ID kosong.
✅ Tidak ada mock tweet.
✅ Semua timestamp valid.
✅ Metadata keyword lengkap.
✅ keyword_count konsisten.
✅ Jumlah ID raw dan final konsisten.
✅ Dataset siap untuk tahap penyimpanan.


True

22. PRODUCTION PRE-FLIGHT CHECK

In [ ]:
# ============================================================
# PRODUCTION PRE-FLIGHT CHECK
# ============================================================

print("=" * 80)
print("PRODUCTION PRE-FLIGHT CHECK")
print("=" * 80)

checks = []


# ============================================================
# 1. DISASTER CASES
# ============================================================

print()
print("[1] DISASTER CASES")

print(
    "Jumlah kasus:",
    len(DISASTER_CASES)
)

if not DISASTER_CASES:

    raise RuntimeError(
        "DISASTER_CASES kosong."
    )


for case_id, config in DISASTER_CASES.items():

    print()
    print(
        f"- {case_id}"
    )

    print(
        "  Label:",
        config["label"]
    )

    print(
        "  Period:",
        config["start_date"],
        "→",
        config["end_date"]
    )

    print(
        "  Event terms:",
        len(config["event_terms"])
    )

    print(
        "  Policy terms:",
        len(config["policy_terms"])
    )

    print(
        "  Query batches:",
        len(config["queries"])
    )


checks.append(
    "✅ Disaster cases tersedia"
)


# ============================================================
# 2. QUERY VALIDATION
# ============================================================

print()
print("[2] QUERY VALIDATION")

for case_id, config in DISASTER_CASES.items():

    if not config["queries"]:

        raise RuntimeError(
            f"Query kosong untuk {case_id}."
        )

    for query in config["queries"]:

        if not query.strip():

            raise RuntimeError(
                f"Query kosong untuk {case_id}."
            )

        if "AND" not in query:

            raise RuntimeError(
                f"Query {case_id} tidak memiliki "
                "kombinasi EVENT AND POLICY."
            )

    print(
        f"  {case_id}: "
        f"{len(config['queries'])} query valid"
    )


checks.append(
    "✅ Research queries valid"
)


# ============================================================
# 3. ADAPTIVE CONFIG
# ============================================================

print()
print("[3] ADAPTIVE CONFIG")

print(
    "Threshold    :",
    ADAPTIVE_THRESHOLD
)

print(
    "Window levels:",
    WINDOW_LEVELS
)

print(
    "Max items    :",
    MAX_ITEMS
)

if (
    ADAPTIVE_THRESHOLD <= 0
    or MAX_ITEMS <= 0
):

    raise RuntimeError(
        "Konfigurasi adaptive tidak valid."
    )

checks.append(
    "✅ Adaptive config valid"
)


# ============================================================
# 4. CHECKPOINT
# ============================================================

checkpoint = (
    load_checkpoint()
)

print()
print("[4] CHECKPOINT")

print(
    "Checkpoint rows:",
    len(checkpoint)
)

duplicate_keys = (
    checkpoint[
        "checkpoint_key"
    ]
    .duplicated()
    .sum()
)

print(
    "Duplicate key:",
    duplicate_keys
)

if duplicate_keys > 0:

    raise RuntimeError(
        "Checkpoint memiliki duplicate key."
    )

checks.append(
    "✅ Checkpoint valid"
)


# ============================================================
# 5. FAILED WINDOWS
# ============================================================

print()
print("[5] FAILED WINDOWS")

if FAILED_FILE.exists():

    failed_df = pd.read_csv(
        FAILED_FILE
    )

    print(
        "Failed windows:",
        len(failed_df)
    )

else:

    failed_df = pd.DataFrame()

    print(
        "Failed windows: 0"
    )


if not failed_df.empty:

    print()
    print(
        failed_df.tail(10)
        .to_string(index=False)
    )

    print()
    print(
        "⚠️ Failed window masih tersedia."
    )

else:

    checks.append(
        "✅ Tidak ada failed window"
    )


# ============================================================
# 6. RAW DIRECTORY
# ============================================================

print()
print("[6] RAW DIRECTORY")

raw_files = sorted(
    RAW_DIR.rglob("*.csv")
)

print(
    "Raw files:",
    len(raw_files)
)

checks.append(
    "✅ Raw directory dapat dibaca"
)


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 80)
print("PRE-FLIGHT RESULT")
print("=" * 80)

for check in checks:
    print(check)

print()
print(
    "🟢 PRE-FLIGHT CHECK BERHASIL."
)

print(
    "Notebook siap menjalankan "
    "FULL SCRAPING."
)

print("=" * 80)

PRODUCTION PRE-FLIGHT CHECK

[1] KEYWORDS
Jumlah keyword: 12
['Palu', 'Donggala', 'gempa Palu', 'gempa Donggala', 'tsunami Palu', 'tsunami Donggala', 'bantuan Palu', 'evakuasi Palu', 'BNPB Palu', 'BPBD Palu', 'relawan Palu', 'logistik Palu']

[2] ADAPTIVE CONFIG
Threshold    : 80
Window levels: {60: 30, 30: 15}
Max items    : 100

[3] CHECKPOINT
Checkpoint rows: 28
Duplicate key  : 0

[4] FAILED WINDOWS
Failed windows: 0

[5] RAW DATA
Raw files: 29

PRE-FLIGHT RESULT
✅ KEYWORDS tersedia
✅ Adaptive config valid
✅ Checkpoint valid
✅ Tidak ada failed window
✅ Raw directory dapat dibaca

🟢 PRE-FLIGHT CHECK BERHASIL.
Notebook siap masuk ke tahap FULL SCRAPING.


23. FULL SCRAPING RUNNER

In [ ]:
# ============================================================
# FULL SCRAPING RUNNER
# ============================================================

def generate_hourly_windows(
    start_date,
    end_date
):
    """
    Membuat window awal 60 menit
    untuk satu periode penelitian.

    Semua waktu menggunakan WIB.
    """

    start = datetime.strptime(
        f"{start_date} 00:00:00",
        "%Y-%m-%d %H:%M:%S"
    )

    end = datetime.strptime(
        f"{end_date} 00:00:00",
        "%Y-%m-%d %H:%M:%S"
    )

    # END DATE dibuat inclusive sampai 23:59:59
    end = end + timedelta(
        days=1
    )

    current = start

    while current < end:

        window_end = min(
            current + timedelta(
                minutes=60
            ),
            end
        )

        yield (
            current.strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
            window_end.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        )

        current = window_end


def run_case(
    case_id,
    config,
    checkpoint
):

    print()
    print("=" * 90)
    print(
        f"START CASE: {case_id}"
    )
    print("=" * 90)

    print(
        "Label :",
        config["label"]
    )

    print(
        "Period:",
        config["start_date"],
        "→",
        config["end_date"]
    )

    queries = config["queries"]

    print(
        "Jumlah query:",
        len(queries)
    )

    total_windows = 0

    for query_index, query in enumerate(
        queries,
        start=1
    ):

        print()
        print("-" * 90)
        print(
            f"QUERY {query_index}/{len(queries)}"
        )
        print("-" * 90)

        print(query)

        windows = list(
            generate_hourly_windows(
                config["start_date"],
                config["end_date"]
            )
        )

        print(
            "Initial windows:",
            len(windows)
        )

        for window_index, (
            start_wib,
            end_wib
        ) in enumerate(
            windows,
            start=1
        ):

            total_windows += 1

            print()
            print(
                f"[{case_id}] "
                f"Query {query_index}/"
                f"{len(queries)} | "
                f"Window "
                f"{window_index}/"
                f"{len(windows)}"
            )

            checkpoint = (
                adaptive_scrape_window(
                    period=case_id,
                    keyword=query,
                    start_wib=start_wib,
                    end_wib=end_wib,
                    window_minutes=60,
                    checkpoint=checkpoint
                )
            )

            # Delay antar request
            time.sleep(
                REQUEST_DELAY_SECONDS
            )

    print()
    print("=" * 90)
    print(
        f"CASE SELESAI: {case_id}"
    )
    print("=" * 90)

    print(
        "Total initial windows:",
        total_windows
    )

    return checkpoint


# ============================================================
# RUN SEMUA KASUS
# ============================================================

checkpoint = load_checkpoint()

for case_id, config in DISASTER_CASES.items():

    checkpoint = run_case(
        case_id=case_id,
        config=config,
        checkpoint=checkpoint
    )


print()
print("=" * 90)
print("FULL SCRAPING SELESAI")
print("=" * 90)

print(
    "Total checkpoint:",
    len(checkpoint)
)

print(
    checkpoint[
        [
            "period",
            "keyword",
            "status",
            "window_minutes",
            "rows_unique"
        ]
    ]
    .tail(20)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# PILOT CONFIGURATION
# ============================================================

PILOT_MODE = True

PILOT_CASES = [
    "palu_2018"
]

PILOT_QUERY_LIMIT = 1

PILOT_HOURS = 1

In [ ]:
# ============================================================
# PILOT / FULL MODE
# ============================================================

if PILOT_MODE:

    selected_cases = {
        case_id: DISASTER_CASES[case_id]
        for case_id in PILOT_CASES
        if case_id in DISASTER_CASES
    }

else:

    selected_cases = DISASTER_CASES


checkpoint = load_checkpoint()


for case_id, config in selected_cases.items():

    config_to_run = config.copy()

    if PILOT_MODE:

        # Ambil hanya query pertama
        config_to_run["queries"] = (
            config["queries"][
                :PILOT_QUERY_LIMIT
            ]
        )

        # Simpan tanggal asli untuk metadata,
        # tetapi pilot dibatasi di runner khusus.
        original_start = (
            config_to_run["start_date"]
        )

        config_to_run["start_date"] = (
            original_start
        )

    checkpoint = run_case(
        case_id=case_id,
        config=config_to_run,
        checkpoint=checkpoint
    )


print()
print(
    "PILOT/FULL SCRAPING SELESAI."
)

23. SAVE DATASET FINAL

In [ ]:
# ============================================================
# SAVE DATASET FINAL
# ============================================================

FINAL_CSV = (
    CLEAN_DIR /
    "bencana_nasional_temporal_clean.csv"
)

FINAL_XLSX = (
    CLEAN_DIR /
    "bencana_nasional_temporal_clean.xlsx"
)


# ============================================================
# CSV
# ============================================================

df_merged.to_csv(
    FINAL_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# EXCEL
# ============================================================

df_merged.to_excel(
    FINAL_XLSX,
    index=False
)


# ============================================================
# OUTPUT
# ============================================================

print("=" * 80)
print("DATASET FINAL TERSIMPAN")
print("=" * 80)

print(
    "CSV   :",
    FINAL_CSV
)

print(
    "Excel :",
    FINAL_XLSX
)

print(
    "Rows  :",
    len(df_merged)
)

print(
    "Cols  :",
    len(df_merged.columns)
)


# ============================================================
# DISTRIBUSI PER KASUS
# ============================================================

if "period" in df_merged.columns:

    print()
    print("=" * 80)
    print("DISTRIBUSI DATA PER KASUS")
    print("=" * 80)

    print(
        df_merged[
            "period"
        ].value_counts()
    )